# 🏋️ ML Coding Drills — Implement ML From Scratch Under Interview Pressure

> **What you'll practise:** 28 timed "implement X from scratch" problems — stable softmax, logistic regression, k-means++, ROC-AUC, backprop, im2col convolution, multi-head attention, top-p sampling, BM25, an LRU cache and more — each with a starter stub, an automatic checker that compares your answer with NumPy / scikit-learn / PyTorch, a hidden solution, complexity analysis, and the follow-up questions interviewers ask next.

| | |
|---|---|
| **Difficulty** | 🟡 Intermediate → 🔴 Advanced |
| **Time** | ~12–15 hours spread over 1–2 weeks (15–30 min per drill) + 3 × 45-minute mock rounds |
| **Prerequisites** | [NumPy](../01_Core_Scientific_Computing/01_NumPy.ipynb) · [ML Fundamentals From Scratch](../03_Classical_Machine_Learning/01_ML_Fundamentals_From_Scratch.ipynb) · [Neural Networks From Scratch](../04_Deep_Learning/01_Neural_Networks_From_Scratch.ipynb) · [Transformers From Scratch](../04_Deep_Learning/03_Transformers_From_Scratch.ipynb) · [Embeddings and Semantic Search](../05_NLP/02_Embeddings_and_Semantic_Search.ipynb) |
| **Tested with** | Python 3.12 · numpy 2.5 · scipy 1.18 · scikit-learn 1.9 · torch 2.14 · torchvision 0.29 · transformers 5.17 · bm25s 0.3 · pandas 3.0 (the libraries are only used to *check* your answers) |
| **Interview relevance** | ⭐⭐⭐ Very high — the "ML coding" round at AI/ML engineer, applied scientist, and research engineer loops: write correct, vectorized, numerically stable code for a core algorithm in 30–45 minutes while explaining it |

## 🎯 How This Round Works

In an **ML coding round** you share an editor (CoderPad, a Google Doc, a Colab/Jupyter notebook, or your own IDE on screen share) and implement an algorithm *from scratch* — usually with only Python + NumPy, sometimes PyTorch tensors (but not `nn.MultiheadAttention` when the task *is* attention). It is different from a LeetCode round: the algorithm is usually well known, so the signal comes from **how cleanly and correctly** you turn math into code.

### Common formats

| Format | What it looks like | Typical length |
|---|---|---|
| **Implement an algorithm** | "Write k-means", "write logistic regression with gradient descent", "write attention" | 30–45 min, 1–2 problems |
| **Vectorize / fix numerics** | "This loop is slow — vectorize it", "this loss returns `nan` — fix it" | 15–20 min |
| **Implement + extend** | Start with a basic version, then the interviewer adds a twist: add a causal mask, handle ties, add L2, make it O(1) | 45 min |
| **ML-flavoured software engineering** | Utilities from AI products: an LRU cache, a rate limiter, chunking, BM25, reciprocal rank fusion | 30–45 min |
| **Take-home** | A few hours; the same skills plus tests, README, and code quality | 2–4 h |

### What interviewers evaluate

| Dimension | What "strong" looks like | Red flags |
|---|---|---|
| **Correctness** | Matches the math; handles the edge cases you listed up front | Off-by-one ranks, wrong axis, forgetting the bias term |
| **Vectorization** | Whole-array operations, `@`, broadcasting, `argpartition` | Triple-nested Python loops over pixels or pairs |
| **Numerical stability** | Shift before `exp`, work in log space, clip before `sqrt`, avoid `inv` | `nan`/`inf` on large logits, `log(0)` |
| **Testing** | A tiny hand-checkable example, asserts on shapes, comparison with a library or brute force | "I think it works" without running anything |
| **Communication** | States assumptions, explains the plan *before* coding, narrates trade-offs | Long silences, or talking without writing code |
| **Complexity** | States time and memory in terms of n, d, k, T and knows the bottleneck | No idea why the O(n²) memory blows up |

### How to think out loud (a script you can reuse)

1. **Clarify:** "Inputs are a `(n, d)` float array and integer labels 0..C−1? Can I use NumPy? Should it handle batches?"
2. **Example:** "Let me write a 2-row example I can check by hand."
3. **Approach:** "I'll compute squared distances with the expansion trick, then `argpartition` — O(n·m) time and memory."
4. **Code:** narrate *why* a line exists ("`keepdims` so the division broadcasts per row"), not *what* it types.
5. **Test:** run the hand example, then an edge case (ties, empty input, huge logits).
6. **Complexity + extensions:** "To scale to 100M vectors I'd switch to an ANN index."

**When you get stuck:** say what you know ("the gradient of softmax + cross-entropy is predictions minus targets"), write the brute-force version first, then optimize. A working O(n²) solution beats an unfinished clever one.

### How to use this notebook

Each drill has: **problem → starter stub (run it: ⏳ = not attempted, ❌ = wrong with a hint, ✅ = correct) → hidden solution with complexity → follow-up questions.** Set a timer (the target time is in each drill's title), write your code in the stub cell, and only open the solution when the checker is green or the timer runs out. Answer the follow-ups **out loud**.

| Section | Drills |
|---|---|
| [1. NumPy Essentials 🟢](#1.-NumPy-Essentials-🟢) | stable softmax/log-softmax · cross-entropy + one-hot · pairwise distances · cosine top-k · moving averages · leak-free standardization |
| [2. Classical ML From Scratch 🟡](#2.-Classical-ML-From-Scratch-🟡) | linear regression · logistic regression · k-means++ · k-NN · decision stump · Gaussian NB · PCA · precision/recall/F1 + ROC-AUC · stratified k-fold |
| [3. Deep Learning Building Blocks 🔴](#3.-Deep-Learning-Building-Blocks-🔴) | MLP backprop + gradient check · conv2d (im2col) · batch norm · inverted dropout · Adam/AdamW · attention + causal mask · temperature/top-k/top-p |
| [4. AI-Engineering Utilities 🟡](#4.-AI-Engineering-Utilities-🟡) | BPE merge · BM25 · reciprocal rank fusion · chunking with overlap · LRU cache · token-bucket rate limiter |
| [🏋️ Timed Drills](#🏋️-Timed-Drills) · [📋 Answer Framework & Cheat Sheet](#📋-Answer-Framework-&-Cheat-Sheet) · [📚 Resources](#📚-Resources) | 3 mock rounds with a scoring rubric, formulas, stable tricks |

## ⚙️ Setup

Run this cell first. Only NumPy is needed to *solve* the drills; SciPy, scikit-learn, PyTorch, transformers, torchvision and bm25s are used by the checkers to compare your answers with trusted implementations. If something is missing, uncomment the `%pip` line, run it once, and restart the kernel.

The `check()` helper gives instant feedback: **✅** correct, **⏳** not attempted yet, **❌** wrong (with a hint).

In [1]:
# %pip install -q "numpy>=2.0" "scipy>=1.13" "scikit-learn>=1.5" "torch>=2.4" torchvision "transformers>=4.45" bm25s pandas

import sys

import numpy as np
import pandas as pd
import scipy
import scipy.special
import scipy.stats
import sklearn
import torch
import torch.nn.functional as F

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | SciPy {scipy.__version__} | "
      f"scikit-learn {sklearn.__version__} | PyTorch {torch.__version__} | pandas {pd.__version__}")

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(0)                       # every drill also creates its own seeded generator, so drills are independent


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    got_arr, exp_arr = np.asarray(got), np.asarray(expected)
    if got_arr.shape != exp_arr.shape:
        raise AssertionError(f"❌ {name}: shape {got_arr.shape}, expected {exp_arr.shape}. {hint}")
    numeric = np.issubdtype(exp_arr.dtype, np.number) and exp_arr.dtype != bool
    ok = np.allclose(got_arr, exp_arr) if numeric else np.array_equal(got_arr, exp_arr)
    assert ok, f"❌ {name}: values are not quite right. {hint}"
    print(f"✅ {name}: correct!")

Python 3.12.11 | NumPy 2.5.3 | SciPy 1.18.1 | scikit-learn 1.9.1 | PyTorch 2.14.0 | pandas 3.0.5


## 1. NumPy Essentials 🟢

These are the warm-ups interviewers use in the first 10 minutes — or as the building blocks of a bigger problem. The recurring theme is **numerical stability**: a `float64` can only hold numbers up to about 1.8 × 10³⁰⁸, so `exp` of anything above ~709.8 overflows to `inf`, and products of many small probabilities underflow to exactly `0.0`. Let's see it happen.

In [2]:
float_max = np.finfo(np.float64).max
print(f"largest float64          : {float_max:.3e}")
print(f"exp overflows above      : {np.log(float_max):.2f}")

with np.errstate(over="ignore", invalid="ignore"):
    big = np.array([1000.0, 1001.0])
    naive_softmax = np.exp(big) / np.exp(big).sum()
print("naive softmax([1000, 1001]):", naive_softmax, "← inf / inf = nan")
print("underflow: np.exp(-800) =", np.exp(-800.0), "| 0.01 ** 200 =", 0.01 ** 200)

largest float64          : 1.798e+308
exp overflows above      : 709.78
naive softmax([1000, 1001]): [nan nan] ← inf / inf = nan
underflow: np.exp(-800) = 0.0 | 0.01 ** 200 = 0.0


### Drill 1.1 — Stable softmax and log-softmax 🟢 (10 min)

**Problem.** Implement `softmax(z, axis=-1)` and `log_softmax(z, axis=-1)` for arrays of any shape.

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \qquad \text{log\_softmax}(z)_i = z_i - \log\sum_j e^{z_j}$$

**Constraints:** NumPy only, no Python loops, must not return `nan`/`inf` for logits like 1000 or −1000, and must respect `axis`.

**Example:** `softmax([1, 2, 3])` → `[0.0900, 0.2447, 0.6652]`.

In [3]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    return None  # TODO: stable softmax along `axis`


def log_softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    return None  # TODO: stable log-softmax along `axis`


Z = np.array([[1.0, 2.0, 3.0],
              [1000.0, 1001.0, 1002.0],
              [-1000.0, 0.0, 1000.0]])
check("softmax vs scipy", softmax(Z), scipy.special.softmax(Z, axis=1),
      hint="Subtract z.max(axis=axis, keepdims=True) before np.exp.")
check("log_softmax vs scipy", log_softmax(Z), scipy.special.log_softmax(Z, axis=1),
      hint="log_softmax = shifted - log(sum(exp(shifted))), where shifted = z - max.")
check("softmax along axis=0", softmax(Z.T, axis=0), scipy.special.softmax(Z.T, axis=0),
      hint="Pass axis to max and sum, always with keepdims=True.")

⏳ softmax vs scipy: not attempted yet — replace None with your answer.
⏳ log_softmax vs scipy: not attempted yet — replace None with your answer.
⏳ softmax along axis=0: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)       # largest logit becomes 0 → exp never overflows
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))   # the log-sum-exp trick


Z = np.array([[1.0, 2.0, 3.0],
              [1000.0, 1001.0, 1002.0],
              [-1000.0, 0.0, 1000.0]])
check("softmax vs scipy", softmax(Z), scipy.special.softmax(Z, axis=1))
check("log_softmax vs scipy", log_softmax(Z), scipy.special.log_softmax(Z, axis=1))
check("softmax along axis=0", softmax(Z.T, axis=0), scipy.special.softmax(Z.T, axis=0))

with np.errstate(divide="ignore"):
    print("np.log(softmax(row 3)):", np.log(softmax(Z[2])), "  ← -inf: information lost")
print("log_softmax(row 3)    :", log_softmax(Z[2]))
```

**Why it works:** multiplying numerator and denominator by $e^{-\max(z)}$ changes nothing mathematically but keeps every exponent ≤ 0.

**Complexity:** O(n) time and O(n) extra memory for n elements.
</details>

#### 🎤 Follow-up questions

**Q1. Doesn't subtracting the maximum change the softmax output?**

<details><summary>Show answer</summary>

- **30-second answer:** No. Softmax is invariant to adding the same constant c to every logit: $e^{z_i - c} / \sum_j e^{z_j - c} = e^{z_i} / \sum_j e^{z_j}$ because $e^{-c}$ cancels. Choosing c = max(z) makes the largest exponent 0, so nothing overflows.
- **Go deeper:** The same invariance is why only *differences* between logits matter, and why a model can drift all logits upward without changing predictions. Subtracting the max doesn't protect against underflow of tiny probabilities to 0 — that's what log-softmax is for.
- **❌ Common wrong answer:** "Subtract the mean" — it still overflows when logits are spread out (e.g. `[-1000, 1000]` has mean 0), or "clip the logits", which changes the answer.

</details>

**Q2. Why do frameworks compute log-softmax directly instead of `log(softmax(z))`?**

<details><summary>Show answer</summary>

- **30-second answer:** Softmax probabilities can underflow to exactly 0, and `log(0) = -inf`, which makes the loss `inf` and gradients `nan`. Log-softmax via log-sum-exp returns the exact finite value (−2000 in the demo above).
- **Go deeper:** That's why PyTorch's `F.cross_entropy` takes **raw logits** and internally does `log_softmax` + negative log-likelihood, and why `nn.BCEWithLogitsLoss` exists next to `nn.BCELoss`. Passing probabilities into `cross_entropy` applies softmax twice — a silent bug.
- **❌ Common wrong answer:** "Add a small epsilon inside the log" — it avoids `-inf` but biases the loss and still loses precision.

</details>

### Drill 1.2 — Cross-entropy from logits with integer labels (+ one-hot) 🟢 (15 min)

**Problem.** Implement:
- `one_hot(y, n_classes)` → `(n, n_classes)` float array,
- `cross_entropy(logits, y)` → the **mean** negative log-likelihood of the true classes, computed from raw logits,
- `cross_entropy_grad(logits, y)` → the gradient of that mean loss with respect to the logits.

**Constraints:** no loops; must be stable for extreme logits; labels are integers `0..C−1`.

**Example:** logits `[[2, 1, 0.1]]`, label `[0]` → loss `0.417`.

In [4]:
logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1000.0, 0.0, -1000.0],
                   [0.0, 0.0, 0.0]])
labels = np.array([0, 1, 2, 2])

t_logits = torch.tensor(logits, requires_grad=True)          # float64 reference from PyTorch
t_loss = F.cross_entropy(t_logits, torch.tensor(labels))
t_loss.backward()
print(f"PyTorch reference loss: {t_loss.item():.4f}")

PyTorch reference loss: 500.4339


In [5]:
def one_hot(y, n_classes):
    return None  # TODO: (n, n_classes) float array


def cross_entropy(logits, y):
    return None  # TODO: mean cross-entropy from raw logits and integer labels


def cross_entropy_grad(logits, y):
    return None  # TODO: d(mean loss) / d(logits), same shape as logits


check("one_hot", one_hot(labels, 3), [[1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1]],
      hint="Start from zeros and set out[np.arange(n), y] = 1.")
check("cross_entropy vs torch", cross_entropy(logits, labels), t_loss.item(),
      hint="Stable log-softmax, then pick each row's true-class log-prob with fancy indexing, negate, average.")
check("gradient vs torch autograd", cross_entropy_grad(logits, labels), t_logits.grad.numpy(),
      hint="(softmax(logits) - one_hot(y)) / n")

⏳ one_hot: not attempted yet — replace None with your answer.
⏳ cross_entropy vs torch: not attempted yet — replace None with your answer.
⏳ gradient vs torch autograd: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def one_hot(y, n_classes):
    y = np.asarray(y)
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def _log_softmax_rows(z):
    shifted = z - z.max(axis=1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))


def cross_entropy(logits, y):
    logp = _log_softmax_rows(np.asarray(logits, dtype=float))
    return float(-logp[np.arange(len(y)), y].mean())


def cross_entropy_grad(logits, y):
    probs = np.exp(_log_softmax_rows(np.asarray(logits, dtype=float)))
    return (probs - one_hot(y, probs.shape[1])) / len(y)


logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [1000.0, 0.0, -1000.0], [0.0, 0.0, 0.0]])
labels = np.array([0, 1, 2, 2])
check("one_hot", one_hot(labels, 3), [[1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1]])
check("cross_entropy vs torch", cross_entropy(logits, labels), t_loss.item())
check("gradient vs torch autograd", cross_entropy_grad(logits, labels), t_logits.grad.numpy())
print(f"loss = {cross_entropy(logits, labels):.3f}  (row 3 alone contributes 2000 / 4 = 500)")
```

**Complexity:** O(n·C) time and memory. Never materialize the one-hot matrix for a 100k-token vocabulary in the loss itself — index with `y` instead (the gradient does need a dense `(n, C)` array, as PyTorch's does).
</details>

#### 🎤 Follow-up questions

**Q3. What is the gradient of softmax cross-entropy with respect to the logits, and why is it so simple?**

<details><summary>Show answer</summary>

- **30-second answer:** For one example, ∂L/∂z = softmax(z) − one_hot(y); for the batch mean divide by n. The Jacobian of softmax and the derivative of −log p combine so all the messy terms cancel.
- **Go deeper:** Derivation: $L = -z_y + \log\sum_j e^{z_j}$, so $\partial L/\partial z_i = -[i=y] + p_i$. The gradient is bounded in [−1, 1] per logit, which is one reason this pairing trains well. The same "prediction minus target" form appears for sigmoid + binary cross-entropy and for linear regression with MSE (up to a factor of 2).
- **❌ Common wrong answer:** Multiplying the softmax Jacobian and the log derivative separately in code — correct in theory but slower and numerically fragile when probabilities are near 0.

</details>

**Q4. Your training loss becomes `nan` after a few hundred steps. Where do you look first?**

<details><summary>Show answer</summary>

- **30-second answer:** Check for `log(0)` or `exp` overflow in a hand-written loss (use log-softmax / `BCEWithLogitsLoss`), a learning rate that's too high (loss spikes before `nan`), division by a zero norm or variance, and bad data (`nan`/`inf` in inputs or labels out of range).
- **Go deeper:** Bisect: `torch.autograd.set_detect_anomaly(True)` for the op that produced it, assert `torch.isfinite` on inputs/activations, log gradient norms, add gradient clipping, and with float16 use loss scaling or bfloat16.
- **❌ Common wrong answer:** "Lower the learning rate until it goes away" without finding the cause — the underlying overflow usually comes back.

</details>

### Drill 1.3 — Pairwise Euclidean distances without loops 🟢 (10 min)

**Problem.** Given `A` with shape `(n, d)` and `B` with shape `(m, d)`, return the `(n, m)` matrix of Euclidean distances.

**Constraints:** no Python loops; memory should be O(n·m), not O(n·m·d); never return `nan`.

**Example:** `A = [[0, 0]]`, `B = [[3, 4], [0, 1]]` → `[[5, 1]]`.

In [6]:
def pairwise_distances(A, B):
    return None  # TODO: (n, m) Euclidean distances, no Python loops


from scipy.spatial.distance import cdist

gen = np.random.default_rng(0)
A = gen.normal(size=(6, 3))
B = gen.normal(size=(4, 3))
big = gen.normal(size=(5, 3)) * 1e4              # large values → larger rounding errors

check("pairwise vs scipy cdist", pairwise_distances(A, B), cdist(A, B),
      hint="‖a‖² + ‖b‖² − 2·a·b using [:, None] and [None, :], then sqrt.")
D_big = pairwise_distances(big, big)
check("no NaN on large self-distances", None if D_big is None else bool(np.isnan(D_big).any()), False,
      hint="Rounding can make a squared distance slightly negative → sqrt gives NaN. Clip at 0 first.")
check("large values still match cdist (atol 1e-3)", None if D_big is None else bool(np.allclose(D_big, cdist(big, big), atol=1e-3)), True)

⏳ pairwise vs scipy cdist: not attempted yet — replace None with your answer.
⏳ no NaN on large self-distances: not attempted yet — replace None with your answer.
⏳ large values still match cdist (atol 1e-3): not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def pairwise_distances(A, B):
    sq = (A ** 2).sum(axis=1)[:, None] + (B ** 2).sum(axis=1)[None, :] - 2.0 * A @ B.T
    return np.sqrt(np.maximum(sq, 0.0))                 # clip tiny negative rounding errors


from scipy.spatial.distance import cdist

gen = np.random.default_rng(0)
A = gen.normal(size=(6, 3))
B = gen.normal(size=(4, 3))
big = gen.normal(size=(5, 3)) * 1e4
check("pairwise vs scipy cdist", pairwise_distances(A, B), cdist(A, B))
D_big = pairwise_distances(big, big)
check("no NaN on large self-distances", bool(np.isnan(D_big).any()), False)
check("large values still match cdist (atol 1e-3)", bool(np.allclose(D_big, cdist(big, big), atol=1e-3)), True)
```

**Complexity:** O(n·m·d) time (dominated by `A @ B.T`), O(n·m) memory. The naive broadcast `A[:, None, :] - B[None, :, :]` is also loop-free but allocates an `(n, m, d)` array — d times more memory.
</details>

#### 🎤 Follow-up questions

**Q5. How are Euclidean distance and cosine similarity related for embeddings?**

<details><summary>Show answer</summary>

- **30-second answer:** For unit-length vectors, ‖a − b‖² = 2 − 2·cos(a, b), so ranking by smallest Euclidean distance equals ranking by largest cosine similarity. Normalize once and you can use a fast inner-product index.
- **Go deeper:** Without normalization, Euclidean distance mixes direction and length; the dot product favours long vectors. Many embedding models are trained with cosine similarity, so normalize unless the model card says otherwise.
- **❌ Common wrong answer:** "They're unrelated metrics, so you must pick the index type carefully" — true only for un-normalized vectors.

</details>

### Drill 1.4 — Cosine top-k retrieval 🟡 (15 min)

**Problem.** Given `queries (q, d)` and `docs (n, d)`, return `(indices, scores)`, both of shape `(q, k)`, holding the k most cosine-similar documents for each query, **best first**.

**Constraints:** no loops over queries or documents; use O(n) selection (`np.argpartition`) rather than a full sort; an all-zero vector must not produce `nan`; must work for `k = 1` and `k = n`.

**Example:** this is exactly the "retrieve" step of semantic search / RAG.

In [7]:
def top_k_cosine(queries, docs, k):
    return None  # TODO: return (indices, scores), both (n_queries, k), best match first


from sklearn.metrics.pairwise import cosine_similarity

gen = np.random.default_rng(1)
docs = gen.normal(size=(50, 8))
queries = gen.normal(size=(3, 8))
reference = cosine_similarity(queries, docs)
ref_idx = np.argsort(-reference, axis=1)

for k in (1, 5, 50):
    result = top_k_cosine(queries, docs, k)
    check(f"top-{k} indices", None if result is None else result[0], ref_idx[:, :k],
          hint="Normalize rows, sims = Qn @ Dn.T, np.argpartition(-sims, k - 1, axis=1)[:, :k], then sort those k.")
    check(f"top-{k} scores", None if result is None else result[1], np.take_along_axis(reference, ref_idx[:, :k], axis=1))
zero_query = top_k_cosine(np.zeros((1, 8)), docs, 3)
check("all-zero query gives finite scores", None if zero_query is None else bool(np.isfinite(zero_query[1]).all()), True,
      hint="Divide by np.maximum(norm, eps) so 0 / 0 never happens.")

⏳ top-1 indices: not attempted yet — replace None with your answer.
⏳ top-1 scores: not attempted yet — replace None with your answer.
⏳ top-5 indices: not attempted yet — replace None with your answer.
⏳ top-5 scores: not attempted yet — replace None with your answer.
⏳ top-50 indices: not attempted yet — replace None with your answer.
⏳ top-50 scores: not attempted yet — replace None with your answer.
⏳ all-zero query gives finite scores: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def top_k_cosine(queries, docs, k, eps=1e-12):
    qn = queries / np.maximum(np.linalg.norm(queries, axis=1, keepdims=True), eps)
    dn = docs / np.maximum(np.linalg.norm(docs, axis=1, keepdims=True), eps)
    sims = qn @ dn.T                                               # (q, n)
    idx = np.argpartition(-sims, k - 1, axis=1)[:, :k]             # k best, in arbitrary order — O(n) per query
    order = np.argsort(-np.take_along_axis(sims, idx, axis=1), axis=1)
    idx = np.take_along_axis(idx, order, axis=1)                   # sort only the k winners
    return idx, np.take_along_axis(sims, idx, axis=1)


from sklearn.metrics.pairwise import cosine_similarity

gen = np.random.default_rng(1)
docs = gen.normal(size=(50, 8))
queries = gen.normal(size=(3, 8))
reference = cosine_similarity(queries, docs)
ref_idx = np.argsort(-reference, axis=1)
for k in (1, 5, 50):
    result = top_k_cosine(queries, docs, k)
    check(f"top-{k} indices", result[0], ref_idx[:, :k])
    check(f"top-{k} scores", result[1], np.take_along_axis(reference, ref_idx[:, :k], axis=1))
zero_query = top_k_cosine(np.zeros((1, 8)), docs, 3)
check("all-zero query gives finite scores", bool(np.isfinite(zero_query[1]).all()), True)
```

**Complexity:** O(q·n·d) to compute similarities, O(q·n) for `argpartition`, O(q·k log k) to sort the winners; O(q·n) memory. For huge `n`, process queries in batches so the `(q, n)` matrix fits in memory.
</details>

#### 🎤 Follow-up questions

**Q6. Brute-force search is too slow for 100 million documents. What do you do?**

<details><summary>Show answer</summary>

- **30-second answer:** Use an approximate nearest-neighbour (ANN) index — HNSW graphs or IVF (cluster then search a few clusters) often combined with product quantization — via FAISS, a vector database, or pgvector. You trade a little recall for orders-of-magnitude lower latency and memory.
- **Go deeper:** Tune recall vs latency (`efSearch` for HNSW, `nprobe` for IVF), measure recall@k against brute force on a sample, compress vectors (float16/int8/binary, Matryoshka truncation), shard the index, and rerank the top candidates with exact scores or a cross-encoder.
- **❌ Common wrong answer:** "Put it on a GPU and keep brute force" — helps constant factors but memory (100M × 768 × 4 bytes ≈ 307 GB) and per-query cost still scale linearly.

</details>

### Drill 1.5 — Simple and exponential moving averages 🟢 (10 min)

**Problem.** Implement:
- `moving_average(x, w)` → the length `len(x) − w + 1` simple moving average ("valid" windows only) in **O(n)** regardless of `w`,
- `ema(x, alpha)` → the exponential moving average $y_0 = x_0$, $y_t = \alpha x_t + (1-\alpha) y_{t-1}$.

**Example:** `moving_average([1, 2, 3, 4], 2)` → `[1.5, 2.5, 3.5]`.

In [8]:
def moving_average(x, w):
    return None  # TODO: simple moving average, 'valid' windows, O(n)


def ema(x, alpha):
    return None  # TODO: exponential moving average with y[0] = x[0]


gen = np.random.default_rng(2)
series = gen.normal(size=30).cumsum()

check("moving_average vs np.convolve", moving_average(series, 5), np.convolve(series, np.ones(5) / 5, mode="valid"),
      hint="c = cumsum of x with a 0 prepended; (c[w:] - c[:-w]) / w.")
check("moving_average with w=1 returns x", moving_average(series, 1), series)
check("ema vs pandas ewm(adjust=False)", ema(series, 0.3), pd.Series(series).ewm(alpha=0.3, adjust=False).mean().to_numpy(),
      hint="A simple loop is fine here: y[t] = alpha * x[t] + (1 - alpha) * y[t-1].")

⏳ moving_average vs np.convolve: not attempted yet — replace None with your answer.
⏳ moving_average with w=1 returns x: not attempted yet — replace None with your answer.
⏳ ema vs pandas ewm(adjust=False): not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def moving_average(x, w):
    c = np.cumsum(np.insert(np.asarray(x, dtype=float), 0, 0.0))
    return (c[w:] - c[:-w]) / w


def ema(x, alpha):
    x = np.asarray(x, dtype=float)
    y = np.empty_like(x)
    y[0] = x[0]
    for t in range(1, len(x)):                     # O(n); each value depends on the previous one
        y[t] = alpha * x[t] + (1 - alpha) * y[t - 1]
    return y


gen = np.random.default_rng(2)
series = gen.normal(size=30).cumsum()
check("moving_average vs np.convolve", moving_average(series, 5), np.convolve(series, np.ones(5) / 5, mode="valid"))
check("moving_average with w=1 returns x", moving_average(series, 1), series)
check("ema vs pandas ewm(adjust=False)", ema(series, 0.3), pd.Series(series).ewm(alpha=0.3, adjust=False).mean().to_numpy())
```

**Complexity:** both O(n) time, O(n) memory. `np.lib.stride_tricks.sliding_window_view(x, w).mean(axis=1)` is the readable alternative but costs O(n·w) time.
</details>

#### 🎤 Follow-up questions

**Q7. Where do exponential moving averages show up in deep learning, and what is "bias correction"?**

<details><summary>Show answer</summary>

- **30-second answer:** Adam's first and second moments, BatchNorm's running mean/variance, and EMA copies of model weights are all EMAs. If an EMA starts at 0 instead of the first value, early estimates are biased toward 0; Adam divides by $1-\beta^t$ to undo that.
- **Go deeper:** With $m_0 = 0$ and a constant input g, $m_t = (1-\beta^t)\,g$, so dividing by $1-\beta^t$ recovers g exactly. An EMA with factor β averages over roughly the last $1/(1-\beta)$ steps (β = 0.999 ≈ 1000 steps).
- **❌ Common wrong answer:** "Bias correction is a regularizer" — it only fixes the initialization bias and matters mostly in the first few thousand steps.

</details>

### Drill 1.6 — Standardization without leakage 🟢 (10 min)

**Problem.** Implement `fit_standardizer(X_train)` → `(mean, std)` and `transform_standardizer(X, mean, std)` → `(X − mean) / std`.

**Constraints:** statistics come **only from the training data**; a constant column (std = 0) must not create `nan`/`inf` (use a scale of 1 there, like scikit-learn).

In [9]:
def fit_standardizer(X_train):
    return None  # TODO: return (mean, std) computed on the TRAINING data only


def transform_standardizer(X, mean, std):
    return None  # TODO


from sklearn.preprocessing import StandardScaler

gen = np.random.default_rng(3)
X_train = gen.normal(loc=50, scale=10, size=(40, 3))
X_train[:, 2] = 7.0                                   # a constant column (std = 0)
X_test = gen.normal(loc=55, scale=12, size=(10, 3))

stats = fit_standardizer(X_train)
scaler = StandardScaler().fit(X_train)
check("train transform vs StandardScaler", None if stats is None else transform_standardizer(X_train, *stats),
      scaler.transform(X_train), hint="std = X_train.std(axis=0); replace zeros with 1.")
check("test transform uses TRAIN statistics", None if stats is None else transform_standardizer(X_test, *stats),
      scaler.transform(X_test), hint="Never call fit on the test set.")

⏳ train transform vs StandardScaler: not attempted yet — replace None with your answer.
⏳ test transform uses TRAIN statistics: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def fit_standardizer(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return mean, np.where(std == 0, 1.0, std)          # constant column → scale 1 (no division by zero)


def transform_standardizer(X, mean, std):
    return (X - mean) / std


from sklearn.preprocessing import StandardScaler

gen = np.random.default_rng(3)
X_train = gen.normal(loc=50, scale=10, size=(40, 3))
X_train[:, 2] = 7.0
X_test = gen.normal(loc=55, scale=12, size=(10, 3))
stats = fit_standardizer(X_train)
scaler = StandardScaler().fit(X_train)
check("train transform vs StandardScaler", transform_standardizer(X_train, *stats), scaler.transform(X_train))
check("test transform uses TRAIN statistics", transform_standardizer(X_test, *stats), scaler.transform(X_test))
print("test column means after scaling:", transform_standardizer(X_test, *stats).mean(axis=0).round(2),
      "← not 0, and that's correct")
```

**Complexity:** O(n·d) time, O(d) memory for the statistics.
</details>

#### 🎤 Follow-up questions

**Q8. Why fit the scaler on the training set only, and how do you do that inside cross-validation and in production?**

<details><summary>Show answer</summary>

- **30-second answer:** Test data must simulate unseen future data; computing mean/std on it leaks information about the test distribution into training and makes scores optimistic. In CV, put the scaler inside a `Pipeline` so it is re-fit on each training fold; in production, save the training statistics with the model and apply them to every request.
- **Go deeper:** For plain standardization the leak is often small, but the same mistake with target encoding, feature selection, imputation using future data, or oversampling before splitting can inflate scores dramatically. Training/serving skew — recomputing statistics differently at serving time — is a classic production bug.
- **❌ Common wrong answer:** "Standardize the whole dataset first, then split — it's just scaling."

</details>

## 2. Classical ML From Scratch 🟡

"Implement logistic regression / k-means / k-NN in NumPy" is the most common ML coding prompt. Interviewers check that you know the **objective**, the **update rule**, the **vectorized** form, and the **edge cases** (empty clusters, ties, zero variance, a class missing from a fold). Each checker below compares your code with scikit-learn on the same data.

> 💡 Review: [ML Fundamentals From Scratch](../03_Classical_Machine_Learning/01_ML_Fundamentals_From_Scratch.ipynb) · [Scikit-Learn](../03_Classical_Machine_Learning/02_Scikit_Learn.ipynb) · [Math for ML](../01_Core_Scientific_Computing/04_Math_for_ML.ipynb)

### Drill 2.1 — Linear regression: closed form and gradient descent 🟢 (15 min)

**Problem.** Implement `fit_linreg_closed_form(X, y)` and `fit_linreg_gd(X, y, lr, epochs)`. Both return a weight vector `[bias, w_1, …, w_d]` minimizing mean squared error $\frac{1}{n}\lVert Xw + b - y\rVert^2$.

**Constraints:** don't call `np.linalg.inv`; gradient descent must be fully vectorized (one matrix product per epoch).

**Test data:** synthetic on purpose (`make_regression`), so we know a linear relationship exists.

In [10]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X_reg, y_reg = make_regression(n_samples=200, n_features=3, noise=10.0, bias=5.0, random_state=0)
sk_linreg = LinearRegression().fit(X_reg, y_reg)
expected_w = np.r_[sk_linreg.intercept_, sk_linreg.coef_]
print("sklearn [bias, w1, w2, w3]:", expected_w.round(4))

sklearn [bias, w1, w2, w3]: [ 4.7236 25.1974 60.3196 57.3446]


In [11]:
def fit_linreg_closed_form(X, y):
    return None  # TODO: weights [bias, w1, ..., wd] via least squares


def fit_linreg_gd(X, y, lr=0.1, epochs=2000):
    return None  # TODO: the same weights via batch gradient descent on the mean squared error


check("closed form vs sklearn", fit_linreg_closed_form(X_reg, y_reg), expected_w,
      hint="Prepend a column of ones, then np.linalg.lstsq(Xb, y, rcond=None).")
check("gradient descent vs sklearn", fit_linreg_gd(X_reg, y_reg), expected_w,
      hint="w -= lr * (2 / n) * Xb.T @ (Xb @ w - y), starting from zeros.")

⏳ closed form vs sklearn: not attempted yet — replace None with your answer.
⏳ gradient descent vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def _with_bias(X):
    return np.column_stack([np.ones(len(X)), X])


def fit_linreg_closed_form(X, y):
    w, *_ = np.linalg.lstsq(_with_bias(X), y, rcond=None)      # SVD-based least squares, never inv(XᵀX)
    return w


def fit_linreg_gd(X, y, lr=0.1, epochs=2000):
    Xb = _with_bias(X)
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * (2.0 / len(y)) * Xb.T @ (Xb @ w - y)           # gradient of the mean squared error
    return w


check("closed form vs sklearn", fit_linreg_closed_form(X_reg, y_reg), expected_w)
check("gradient descent vs sklearn", fit_linreg_gd(X_reg, y_reg), expected_w)

Xb = _with_bias(X_reg)
hessian = (2.0 / len(y_reg)) * Xb.T @ Xb
print(f"GD is stable only for lr < 2 / λ_max = {2 / np.linalg.eigvalsh(hessian).max():.3f} (we used 0.1)")
print(f"cond(Xb) = {np.linalg.cond(Xb):.4f} | cond(XbᵀXb) = {np.linalg.cond(Xb.T @ Xb):.4f} | cond(Xb)² = {np.linalg.cond(Xb) ** 2:.4f}")
```

**Complexity:** closed form O(n·d² + d³) time, O(n·d) memory; gradient descent O(epochs · n·d) time, O(n·d) memory. For millions of rows use mini-batch SGD.
</details>

#### 🎤 Follow-up questions

**Q9. Why use `lstsq` or `solve` instead of the textbook formula $w = (X^\top X)^{-1} X^\top y$?**

<details><summary>Show answer</summary>

- **30-second answer:** Forming $X^\top X$ squares the condition number (the solution cell prints cond(XᵀX) = cond(X)²), so small rounding errors get amplified, and explicitly inverting is slower and less accurate than a factorization. `lstsq` works on X directly with SVD, and also handles rank-deficient X.
- **Go deeper:** With highly correlated features (multicollinearity) $X^\top X$ is nearly singular; ridge regression adds $\lambda I$, which makes it well-conditioned: $w = (X^\top X + \lambda I)^{-1} X^\top y$ — solved with `np.linalg.solve`, not `inv`, and without penalizing the bias.
- **❌ Common wrong answer:** "`inv` is fine because NumPy uses float64."

</details>

**Q10. Your gradient descent diverges (loss → ∞) or crawls. What's going on, and how do you pick the learning rate?**

<details><summary>Show answer</summary>

- **30-second answer:** For MSE the loss is a quadratic bowl with Hessian $H = \frac{2}{n}X^\top X$; gradient descent converges only if lr < 2 / λ_max(H). Unscaled features make λ_max huge (forcing a tiny lr) while λ_min stays small, so progress along flat directions crawls. Standardize features, then tune lr on a log scale.
- **Go deeper:** Convergence speed depends on the condition number λ_max / λ_min. Momentum, Adam, or second-order methods help with ill-conditioning; a learning-rate finder (increase lr until loss explodes) gives a quick upper bound.
- **❌ Common wrong answer:** "It diverges because the data isn't linear" — a linear model on non-linear data still converges; it just underfits.

</details>

### Drill 2.2 — Logistic regression with L2 regularization 🟡 (20 min)

**Problem.** Implement a numerically stable `sigmoid(z)` and `fit_logreg(X, y, lr, epochs, l2)` returning `(w, b)` that minimize

$$\frac{1}{n}\sum_i \Big[\log(1 + e^{z_i}) - y_i z_i\Big] + \frac{\lambda}{2}\lVert w\rVert^2, \qquad z_i = w^\top x_i + b$$

(the bias is not penalized).

**Constraints:** no overflow warnings for |z| = 1000; vectorized gradient.

**Matching scikit-learn:** `LogisticRegression(C)` minimizes $\frac{1}{2}\lVert w\rVert^2 + C\sum_i \text{loss}_i$. Dividing by $C\cdot n$ gives our objective with $\lambda = 1/(C\cdot n)$.

In [12]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# synthetic on purpose: 10% of labels are flipped, so the classes are NOT perfectly separable
X_clf, y_clf = make_classification(n_samples=300, n_features=4, n_informative=3, n_redundant=0,
                                   flip_y=0.1, random_state=0)
X_clf = (X_clf - X_clf.mean(axis=0)) / X_clf.std(axis=0)
C = 1.0
sk_logreg = LogisticRegression(C=C, max_iter=10_000, tol=1e-12).fit(X_clf, y_clf)
print("sklearn weights:", sk_logreg.coef_[0].round(4), "| bias:", round(float(sk_logreg.intercept_[0]), 4))

sklearn weights: [ 1.2113  0.4316 -0.0434 -0.722 ] | bias: -0.0511


In [13]:
def sigmoid(z):
    return None  # TODO: numerically stable sigmoid


def fit_logreg(X, y, lr=0.5, epochs=1000, l2=0.0):
    return None  # TODO: return (w, b) minimizing mean log-loss + (l2 / 2) * ||w||²


check("stable sigmoid", sigmoid(np.array([-1000.0, 0.0, 1000.0])), [0.0, 0.5, 1.0],
      hint="For z < 0 compute exp(z) / (1 + exp(z)) so exp never overflows.")
fit = fit_logreg(X_clf, y_clf, l2=1.0 / (C * len(y_clf)))
check("weights vs sklearn (C = 1)", None if fit is None else fit[0], sk_logreg.coef_[0],
      hint="grad_w = X.T @ (p - y) / n + l2 * w; grad_b = mean(p - y).")
check("bias vs sklearn", None if fit is None else fit[1], sk_logreg.intercept_[0])

⏳ stable sigmoid: not attempted yet — replace None with your answer.
⏳ weights vs sklearn (C = 1): not attempted yet — replace None with your answer.
⏳ bias vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))          # exp of a non-positive number: safe
    ez = np.exp(z[~pos])                              # z < 0 here, so exp(z) < 1: safe
    out[~pos] = ez / (1.0 + ez)
    return out


def fit_logreg(X, y, lr=0.5, epochs=1000, l2=0.0):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    for _ in range(epochs):
        error = sigmoid(X @ w + b) - y                 # (n,) — "prediction minus target"
        w -= lr * (X.T @ error / n + l2 * w)
        b -= lr * error.mean()
    return w, b


check("stable sigmoid", sigmoid(np.array([-1000.0, 0.0, 1000.0])), [0.0, 0.5, 1.0])
fit = fit_logreg(X_clf, y_clf, l2=1.0 / (C * len(y_clf)))
check("weights vs sklearn (C = 1)", fit[0], sk_logreg.coef_[0])
check("bias vs sklearn", fit[1], sk_logreg.intercept_[0])

z = X_clf @ fit[0] + fit[1]
print(f"mean log-loss (stable form np.logaddexp(0, z) - y*z): {np.mean(np.logaddexp(0.0, z) - y_clf * z):.4f}")
```

`scipy.special.expit` is the library's stable sigmoid. For the loss, `np.logaddexp(0, z) - y·z` never takes `log(0)`.

**Complexity:** O(epochs · n·d) time, O(n + d) extra memory.
</details>

#### 🎤 Follow-up questions

**Q11. Why train logistic regression with log-loss instead of mean squared error?**

<details><summary>Show answer</summary>

- **30-second answer:** Log-loss is the negative log-likelihood of a Bernoulli model, so minimizing it is maximum likelihood estimation; it is convex in the weights, and its gradient $(p - y)\,x$ stays large when the model is confidently wrong. MSE on a sigmoid output is non-convex in w, and its gradient contains σ′(z), which vanishes when the sigmoid saturates — so confident mistakes learn slowly.
- **Go deeper:** Log-loss also yields probabilities that tend to be reasonably calibrated for a well-specified model, and it heavily penalizes confident errors (−log p → ∞ as p → 0), which is why label noise can hurt it.
- **❌ Common wrong answer:** "MSE can't be used for classification at all" — it can; it's just a worse objective here.

</details>

**Q12. What happens if the training data is perfectly separable and you use no regularization?**

<details><summary>Show answer</summary>

- **30-second answer:** There is no finite optimum: scaling w up always lowers the loss, so the weights grow without bound, predicted probabilities go to exactly 0 or 1, and the model becomes overconfident. Regularization (L2, which scikit-learn applies by default with C = 1.0) or early stopping gives a finite, better-behaved solution.
- **Go deeper:** Plain gradient descent on separable data converges in *direction* to the maximum-margin (hard-margin SVM) separator even as the norm diverges (Soudry et al., 2018). In scikit-learn 1.9 you request "no penalty" with `C=np.inf`; the `penalty` argument was deprecated in 1.8.
- **❌ Common wrong answer:** "You get a perfect model with finite weights."

</details>

### Drill 2.3 — k-means with k-means++ initialization 🟡 (25 min)

**Problem.** Implement `kmeans_plus_plus_init(X, k, rng)` and `kmeans(X, k, rng, n_iter)` returning `(centers, labels, inertia)`, where inertia is the sum of squared distances from each point to its assigned center.

**k-means++:** pick the first center uniformly at random; pick each next center with probability proportional to $D(x)^2$, the squared distance from x to its nearest already-chosen center.

**Constraints:** vectorized distance computation; handle a cluster that becomes empty (keep its old center); stop early when centers stop moving.

In [14]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=3)   # synthetic: 4 known clusters
sk_kmeans = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_blobs)
print(f"sklearn inertia: {sk_kmeans.inertia_:.3f}")

sklearn inertia: 397.632


In [15]:
def kmeans_plus_plus_init(X, k, rng):
    return None  # TODO: (k, d) initial centers chosen with D² sampling


def kmeans(X, k, rng, n_iter=100):
    return None  # TODO: return (centers, labels, inertia)


init = kmeans_plus_plus_init(X_blobs, 4, np.random.default_rng(0))
check("init returns k distinct rows of X",
      None if init is None else bool(len(np.unique(init, axis=0)) == 4 and all((X_blobs == c).all(axis=1).any() for c in init)),
      True, hint="Centers must be actual data points: X[rng.choice(n, p=d2 / d2.sum())].")
result = kmeans(X_blobs, 4, np.random.default_rng(0))
check("same partition as sklearn (adjusted Rand index = 1)", None if result is None else adjusted_rand_score(sk_kmeans.labels_, result[1]), 1.0,
      hint="Alternate: assign each point to its nearest center; move each center to the mean of its points.")
check("inertia vs sklearn", None if result is None else result[2], sk_kmeans.inertia_)

⏳ init returns k distinct rows of X: not attempted yet — replace None with your answer.
⏳ same partition as sklearn (adjusted Rand index = 1): not attempted yet — replace None with your answer.
⏳ inertia vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def kmeans_plus_plus_init(X, k, rng):
    centers = [X[rng.integers(len(X))]]                                   # first center: uniform
    for _ in range(1, k):
        d2 = ((X[:, None, :] - np.array(centers)[None, :, :]) ** 2).sum(axis=2).min(axis=1)
        centers.append(X[rng.choice(len(X), p=d2 / d2.sum())])            # next center: probability ∝ D(x)²
    return np.array(centers)


def kmeans(X, k, rng, n_iter=100, tol=1e-10):
    centers = kmeans_plus_plus_init(X, k, rng)
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)     # (n, k) squared distances
        labels = d2.argmin(axis=1)                                         # assignment step
        new_centers = np.array([X[labels == j].mean(axis=0) if np.any(labels == j) else centers[j]
                                for j in range(k)])                        # update step; empty cluster keeps its center
        shift = ((new_centers - centers) ** 2).sum()
        centers = new_centers
        if shift < tol:
            break
    d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = d2.argmin(axis=1)
    return centers, labels, float(d2[np.arange(len(X)), labels].sum())


init = kmeans_plus_plus_init(X_blobs, 4, np.random.default_rng(0))
check("init returns k distinct rows of X",
      bool(len(np.unique(init, axis=0)) == 4 and all((X_blobs == c).all(axis=1).any() for c in init)), True)
result = kmeans(X_blobs, 4, np.random.default_rng(0))
check("same partition as sklearn (adjusted Rand index = 1)", adjusted_rand_score(sk_kmeans.labels_, result[1]), 1.0)
check("inertia vs sklearn", result[2], sk_kmeans.inertia_)
```

Cluster ids are arbitrary (cluster 0 in ours may be cluster 2 in sklearn's), which is why the checker uses the adjusted Rand index instead of comparing labels directly.

**Complexity:** each Lloyd iteration is O(n·k·d) time; the broadcast above uses O(n·k·d) memory (use the ‖a‖² + ‖b‖² − 2a·b trick for O(n·k)). This k-means++ recomputes distances to all chosen centers, O(n·k²·d) in total; keeping a running minimum makes it O(n·k·d).
</details>

#### 🎤 Follow-up questions

**Q13. Why use k-means++ initialization instead of picking k random points?**

<details><summary>Show answer</summary>

- **30-second answer:** Random starts often place two centers in the same true cluster, and Lloyd's algorithm only finds a local optimum, so it gets stuck. D² sampling spreads the initial centers out; Arthur & Vassilvitskii (2007) proved the expected cost is within O(log k) of optimal, and in practice it converges faster to lower inertia.
- **Go deeper:** It is still randomized, so libraries run several initializations (`n_init`) and keep the lowest inertia. scikit-learn uses a "greedy" variant that samples several candidates per step and keeps the one that reduces inertia most.
- **❌ Common wrong answer:** "k-means++ guarantees the global optimum" — k-means is NP-hard; the guarantee is approximate and in expectation.

</details>

**Q14. How do you choose k, and when is k-means the wrong tool?**

<details><summary>Show answer</summary>

- **30-second answer:** Use the elbow of inertia vs k, the silhouette score, stability across resamples, or — best — a downstream metric or business constraint. k-means assumes roughly spherical, similar-sized clusters in Euclidean space, is sensitive to feature scale and outliers, and always returns k clusters even if the data has none.
- **Go deeper:** Gaussian mixtures handle elliptical clusters and give soft assignments; DBSCAN/HDBSCAN find arbitrary shapes and label noise; for millions of points use mini-batch k-means. Always standardize features first.
- **❌ Common wrong answer:** "Pick the k with the lowest inertia" — inertia always decreases as k grows (it reaches 0 at k = n).

</details>

### Drill 2.4 — k-nearest neighbours classifier 🟢 (15 min)

**Problem.** Implement `knn_predict(X_train, y_train, X_query, k)` using Euclidean distance and a majority vote; break vote ties by choosing the **smallest** label (scikit-learn's behaviour).

**Constraints:** no loops over queries; O(n) neighbour selection with `argpartition`.

**Test data:** the real UCI wine dataset (178 wines, 13 chemical measurements, 3 cultivars), standardized with **training** statistics.

In [16]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier

wine = load_wine()
perm = np.random.default_rng(1).permutation(len(wine.target))
tr_idx, te_idx = perm[:120], perm[120:]
mu_w, sd_w = wine.data[tr_idx].mean(axis=0), wine.data[tr_idx].std(axis=0)
Xw_tr, Xw_te = (wine.data[tr_idx] - mu_w) / sd_w, (wine.data[te_idx] - mu_w) / sd_w
yw_tr, yw_te = wine.target[tr_idx], wine.target[te_idx]
sk_knn_pred = KNeighborsClassifier(n_neighbors=5).fit(Xw_tr, yw_tr).predict(Xw_te)
print(f"sklearn k-NN test accuracy: {(sk_knn_pred == yw_te).mean():.3f}")

sklearn k-NN test accuracy: 0.948


In [17]:
def knn_predict(X_train, y_train, X_query, k=5):
    return None  # TODO: majority vote of the k nearest training points (ties → smallest label)


knn_pred = knn_predict(Xw_tr, yw_tr, Xw_te, k=5)
check("predictions vs sklearn", knn_pred, sk_knn_pred,
      hint="Squared distances with the expansion trick → argpartition → count votes per class → argmax.")

⏳ predictions vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def knn_predict(X_train, y_train, X_query, k=5):
    d2 = (X_query ** 2).sum(axis=1)[:, None] + (X_train ** 2).sum(axis=1)[None, :] - 2.0 * X_query @ X_train.T
    nearest = np.argpartition(d2, k - 1, axis=1)[:, :k]                     # (q, k) neighbour indices, unordered
    votes = y_train[nearest]                                                 # (q, k) neighbour labels
    n_classes = int(y_train.max()) + 1
    counts = (votes[:, :, None] == np.arange(n_classes)[None, None, :]).sum(axis=1)   # (q, C) vote counts
    return counts.argmax(axis=1)                                             # argmax returns the first (smallest) label on ties


knn_pred = knn_predict(Xw_tr, yw_tr, Xw_te, k=5)
check("predictions vs sklearn", knn_pred, sk_knn_pred)
print(f"our k-NN test accuracy: {(knn_pred == yw_te).mean():.3f}")
```

Squared distances give the same neighbours as distances (sqrt is monotonic), so we skip the sqrt.

**Complexity:** "training" is O(1) (just store the data); prediction is O(q·n·d) time and O(q·n) memory. Batch the queries if `q·n` is too large.
</details>

#### 🎤 Follow-up question

**Q15. What are the costs and failure modes of k-NN, and how do you make it fast?**

<details><summary>Show answer</summary>

- **30-second answer:** Training is free, but every prediction scans all n training points (O(n·d)) and the model stores the whole dataset. It needs feature scaling (distance mixes units) and degrades in high dimensions, where distances concentrate and "nearest" neighbours are barely nearer than random points. Small k → high variance; large k → high bias.
- **Go deeper:** Speed-ups: KD-trees/ball trees for low dimensions, approximate nearest-neighbour indexes (HNSW, IVF-PQ) for high dimensions, dimensionality reduction (PCA), or condensing the training set. Distance weighting (`weights="distance"`) reduces sensitivity to k.
- **❌ Common wrong answer:** "k-NN is slow to train."

</details>

### Drill 2.5 — Best split for a decision stump (Gini and entropy) 🟡 (25 min)

**Problem.** Implement `gini(counts)`, `entropy(counts)` (in bits), and `best_split(X, y, criterion)` returning `(feature, threshold, weighted_child_impurity)` for the single best axis-aligned split `x[feature] <= threshold`.

$$\text{Gini} = 1 - \sum_c p_c^2, \qquad H = -\sum_c p_c \log_2 p_c, \qquad \text{weighted} = \tfrac{n_L}{n} I_L + \tfrac{n_R}{n} I_R$$

**Constraints:** thresholds are midpoints between consecutive distinct sorted values (as in scikit-learn); aim for O(d · n log n) by sorting each feature once and using cumulative class counts, not O(d · n²).

**Test data:** the real breast cancer dataset (569 tumours, 30 features).

In [18]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

cancer = load_breast_cancer()
stump_refs = {}
for criterion in ("gini", "entropy"):
    tree = DecisionTreeClassifier(max_depth=1, criterion=criterion, random_state=0).fit(cancer.data, cancer.target).tree_
    weighted = (tree.n_node_samples[1] * tree.impurity[1] + tree.n_node_samples[2] * tree.impurity[2]) / tree.n_node_samples[0]
    stump_refs[criterion] = (int(tree.feature[0]), float(tree.threshold[0]), float(weighted))
    print(f"sklearn {criterion:7s}: split on feature {tree.feature[0]} ({cancer.feature_names[tree.feature[0]]}) "
          f"<= {tree.threshold[0]:.4f}, weighted child impurity {weighted:.4f}")

sklearn gini   : split on feature 20 (worst radius) <= 16.7950, weighted child impurity 0.1423
sklearn entropy: split on feature 22 (worst perimeter) <= 105.9500, weighted child impurity 0.3906


In [19]:
def gini(counts):
    return None  # TODO: Gini impurity from class counts


def entropy(counts):
    return None  # TODO: entropy in bits from class counts


def best_split(X, y, criterion="gini"):
    return None  # TODO: return (feature, threshold, weighted child impurity)


check("gini([5, 5])", gini(np.array([5, 5])), 0.5)
check("entropy([5, 5])", entropy(np.array([5, 5])), 1.0)
check("entropy([10, 0]) is 0 (pure node)", entropy(np.array([10, 0])), 0.0, hint="Skip zero counts: 0·log 0 is defined as 0.")
for criterion, (ref_feature, ref_threshold, ref_weighted) in stump_refs.items():
    split = best_split(cancer.data, cancer.target, criterion)
    check(f"{criterion}: feature", None if split is None else split[0], ref_feature)
    check(f"{criterion}: threshold", None if split is None else split[1], ref_threshold,
          hint="Midpoint between consecutive DISTINCT sorted values.")
    check(f"{criterion}: weighted child impurity", None if split is None else split[2], ref_weighted)

⏳ gini([5, 5]): not attempted yet — replace None with your answer.
⏳ entropy([5, 5]): not attempted yet — replace None with your answer.
⏳ entropy([10, 0]) is 0 (pure node): not attempted yet — replace None with your answer.
⏳ gini: feature: not attempted yet — replace None with your answer.
⏳ gini: threshold: not attempted yet — replace None with your answer.
⏳ gini: weighted child impurity: not attempted yet — replace None with your answer.
⏳ entropy: feature: not attempted yet — replace None with your answer.
⏳ entropy: threshold: not attempted yet — replace None with your answer.
⏳ entropy: weighted child impurity: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def gini(counts):
    p = np.asarray(counts, dtype=float) / np.sum(counts)
    return float(1.0 - np.sum(p ** 2))


def entropy(counts):
    p = np.asarray(counts, dtype=float) / np.sum(counts)
    p = p[p > 0]                                               # 0·log 0 = 0
    return float(-np.sum(p * np.log2(p)))


def _impurity_rows(counts, criterion):
    p = counts / counts.sum(axis=1, keepdims=True)            # one row per candidate split
    if criterion == "gini":
        return 1.0 - (p ** 2).sum(axis=1)
    return -(p * np.log2(np.where(p > 0, p, 1.0))).sum(axis=1)   # log2(1) = 0 handles empty classes


def best_split(X, y, criterion="gini"):
    n, d = X.shape
    classes = np.unique(y)
    best = (None, None, np.inf)
    for j in range(d):                                         # O(d · n log n) overall
        order = np.argsort(X[:, j], kind="stable")
        xs, ys = X[order, j], y[order]
        onehot = (ys[:, None] == classes[None, :]).astype(float)
        left = np.cumsum(onehot, axis=0)[:-1]                  # row i: class counts of the first i+1 samples
        right = onehot.sum(axis=0) - left
        n_left = np.arange(1, n)
        weighted = (n_left * _impurity_rows(left, criterion) + (n - n_left) * _impurity_rows(right, criterion)) / n
        weighted[xs[1:] == xs[:-1]] = np.inf                   # can't split between two equal values
        i = int(np.argmin(weighted))
        if weighted[i] < best[2]:
            best = (j, float((xs[i] + xs[i + 1]) / 2), float(weighted[i]))
    return best


check("gini([5, 5])", gini(np.array([5, 5])), 0.5)
check("entropy([5, 5])", entropy(np.array([5, 5])), 1.0)
check("entropy([10, 0]) is 0 (pure node)", entropy(np.array([10, 0])), 0.0)
for criterion, (ref_feature, ref_threshold, ref_weighted) in stump_refs.items():
    split = best_split(cancer.data, cancer.target, criterion)
    check(f"{criterion}: feature", split[0], ref_feature)
    check(f"{criterion}: threshold", split[1], ref_threshold)
    check(f"{criterion}: weighted child impurity", split[2], ref_weighted)
```

**Complexity:** O(d · n log n) time for the sorts plus O(d · n · C) for the cumulative counts; O(n · C) memory per feature. A full tree repeats this at every node.
</details>

#### 🎤 Follow-up questions

**Q16. Gini or entropy — does the choice matter?**

<details><summary>Show answer</summary>

- **30-second answer:** Rarely. Both are 0 for pure nodes, maximal for uniform classes, and usually lead to trees of very similar quality. On the breast-cancer stumps above they chose different but closely related features (the printout shows "worst radius" for Gini and "worst perimeter" for entropy). Gini avoids a logarithm, so it's slightly cheaper and is scikit-learn's default.
- **Go deeper:** For two classes, Gini = 2p(1 − p) and entropy = −p log₂ p − (1 − p) log₂(1 − p); both are concave with the same maximum at p = 0.5. Depth, `min_samples_leaf`, and ensembling matter far more than the criterion.
- **❌ Common wrong answer:** "Entropy is always more accurate because it comes from information theory."

</details>

**Q17. How do XGBoost and LightGBM find splits quickly on millions of rows?**

<details><summary>Show answer</summary>

- **30-second answer:** Histogram-based splitting: each feature is bucketed once into at most ~255 bins, then each node accumulates gradient and Hessian sums per bin and scans the bins — O(bins) per feature per node instead of re-sorting O(n log n) values. The split gain is computed from those sums, e.g. $\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}$ (up to a constant factor and a complexity penalty γ).
- **Go deeper:** LightGBM adds gradient-based one-side sampling (GOSS), exclusive feature bundling and leaf-wise growth; XGBoost's default `tree_method` has been `"hist"` since 2.0; a child's histogram can be obtained by subtracting its sibling's from the parent's.
- **❌ Common wrong answer:** "They sort every feature at every node, just in C++."

</details>

### Drill 2.6 — Gaussian Naive Bayes 🟡 (20 min)

**Problem.** Implement `gnb_fit(X, y, var_smoothing)` and `gnb_predict_proba(model, X)`. Each class c has a prior $P(c)$ and, per feature, a Gaussian with its own mean and variance; the posterior is

$$P(c \mid x) \propto P(c) \prod_j \mathcal{N}(x_j \mid \mu_{cj}, \sigma^2_{cj})$$

**Constraints:** compute in **log space** and normalize with log-sum-exp; add `var_smoothing × (largest feature variance of X)` to every variance, exactly like scikit-learn.

**Test data:** the real iris dataset, stratified 70/30 split.

In [20]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

iris = load_iris()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(iris.data, iris.target, test_size=0.3, stratify=iris.target, random_state=0)
sk_nb = GaussianNB().fit(Xi_tr, yi_tr)
print(f"sklearn GaussianNB test accuracy: {sk_nb.score(Xi_te, yi_te):.3f}")

sklearn GaussianNB test accuracy: 0.978


In [21]:
def gnb_fit(X, y, var_smoothing=1e-9):
    return None  # TODO: return a dict with classes, log priors, means (C, d) and variances (C, d)


def gnb_predict_proba(model, X):
    return None  # TODO: (n, C) posterior probabilities, computed in log space


nb_model = gnb_fit(Xi_tr, yi_tr)
check("posteriors vs sklearn", None if nb_model is None else gnb_predict_proba(nb_model, Xi_te), sk_nb.predict_proba(Xi_te),
      hint="log N(x | μ, σ²) = −½[log(2πσ²) + (x − μ)²/σ²]; sum over features, add the log prior, subtract logsumexp per row.")

⏳ posteriors vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def gnb_fit(X, y, var_smoothing=1e-9):
    classes = np.unique(y)
    eps = var_smoothing * X.var(axis=0).max()                            # same stabilizer as scikit-learn
    return {
        "classes": classes,
        "log_prior": np.log(np.array([np.mean(y == c) for c in classes])),
        "mean": np.array([X[y == c].mean(axis=0) for c in classes]),     # (C, d)
        "var": np.array([X[y == c].var(axis=0) for c in classes]) + eps,  # (C, d)
    }


def gnb_predict_proba(model, X):
    mean, var = model["mean"][None], model["var"][None]                 # (1, C, d)
    log_lik = -0.5 * (np.log(2 * np.pi * var) + (X[:, None, :] - mean) ** 2 / var).sum(axis=2)   # (n, C)
    joint = log_lik + model["log_prior"][None, :]
    return np.exp(joint - scipy.special.logsumexp(joint, axis=1, keepdims=True))


nb_model = gnb_fit(Xi_tr, yi_tr)
check("posteriors vs sklearn", gnb_predict_proba(nb_model, Xi_te), sk_nb.predict_proba(Xi_te))

likelihoods = np.full(1000, 0.01)
print("product of 1000 likelihoods of 0.01:", np.prod(likelihoods), "| sum of their logs:", round(float(np.log(likelihoods).sum()), 1))
```

**Complexity:** fitting O(n·d) time, O(C·d) memory; prediction O(n·C·d) time and memory.
</details>

#### 🎤 Follow-up question

**Q18. What is "naive" about Naive Bayes, why does it still work, and why compute in log space?**

<details><summary>Show answer</summary>

- **30-second answer:** It assumes features are **conditionally independent given the class**, so the likelihood factorizes into a product of one-dimensional densities. The assumption is usually false, but classification only needs the *highest* posterior to be right, so it often works well — especially with little data or many features (spam filtering). Log space turns the product into a sum: multiplying 1000 likelihoods of 0.01 underflows to exactly 0.0 (demo in the solution), while the sum of logs is a normal number.
- **Go deeper:** Because correlated features are double-counted, NB's probabilities are often overconfident — calibrate them (Platt scaling / isotonic) if you need probabilities. Multinomial/Bernoulli NB with Laplace smoothing are the text-classification variants.
- **❌ Common wrong answer:** "It assumes the features are independent" — the assumption is independence *conditional on the class*, which is different.

</details>

### Drill 2.7 — PCA via SVD 🟡 (15 min)

**Problem.** Implement `pca_svd(X, n_components)` returning `(components, explained_variance, explained_variance_ratio, X_projected)`.

**Recipe:** center X; $X_c = U\,\text{diag}(S)\,V^\top$; the components are the first rows of $V^\top$; explained variance is $S^2/(n-1)$; the ratio divides by the sum over **all** components; the projection is $X_c V_k$.

**Test data:** the real wine dataset, standardized (its features have different units).

In [22]:
from sklearn.decomposition import PCA

wine_X = load_wine().data
wine_X = (wine_X - wine_X.mean(axis=0)) / wine_X.std(axis=0)      # different units → standardize before PCA
sk_pca = PCA(n_components=3).fit(wine_X)
print("sklearn explained variance ratio:", sk_pca.explained_variance_ratio_.round(4))

sklearn explained variance ratio: [0.362  0.1921 0.1112]


In [23]:
def pca_svd(X, n_components):
    return None  # TODO: return (components, explained_variance, explained_variance_ratio, X_projected)


pca_out = pca_svd(wine_X, 3)
if pca_out is None:
    check("PCA via SVD", None, sk_pca.components_)
else:
    comps, ev, evr, Z_pca = pca_out
    signs = np.sign(np.sum(comps * sk_pca.components_, axis=1))    # each component is only defined up to ±1
    check("components (up to sign)", comps * signs[:, None], sk_pca.components_,
          hint="Center X, U, S, Vt = np.linalg.svd(Xc, full_matrices=False), components = Vt[:k].")
    check("explained variance", ev, sk_pca.explained_variance_, hint="S² / (n − 1)")
    check("explained variance ratio", evr, sk_pca.explained_variance_ratio_, hint="Divide by the sum over ALL singular values.")
    check("projected data (up to sign)", Z_pca * signs, sk_pca.transform(wine_X), hint="Xc @ components.T")

⏳ PCA via SVD: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def pca_svd(X, n_components):
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)              # Xc = U · diag(S) · Vt
    components = Vt[:n_components]                                  # rows = principal directions
    explained_variance = S ** 2 / (len(X) - 1)
    ratio = explained_variance / explained_variance.sum()
    return components, explained_variance[:n_components], ratio[:n_components], Xc @ components.T


comps, ev, evr, Z_pca = pca_svd(wine_X, 3)
signs = np.sign(np.sum(comps * sk_pca.components_, axis=1))
check("components (up to sign)", comps * signs[:, None], sk_pca.components_)
check("explained variance", ev, sk_pca.explained_variance_)
check("explained variance ratio", evr, sk_pca.explained_variance_ratio_)
check("projected data (up to sign)", Z_pca * signs, sk_pca.transform(wine_X))
```

**Complexity:** a thin SVD of an n × d matrix costs O(n·d·min(n, d)) time and O(n·d) memory. For only the top-k components of a huge matrix, randomized SVD is much cheaper.
</details>

#### 🎤 Follow-up questions

**Q19. Why compute PCA with an SVD of the data instead of an eigendecomposition of the covariance matrix?**

<details><summary>Show answer</summary>

- **30-second answer:** They give the same answer — the covariance is $V\,\text{diag}(S^2)\,V^\top/(n-1)$ — but forming $X^\top X$ squares the condition number, so small-variance components lose accuracy. SVD works on X directly, is numerically stabler, and handles d > n naturally.
- **Go deeper:** When n ≫ d and d is small, the covariance route is faster (a d × d eigenproblem), and scikit-learn's `svd_solver="auto"` picks between full SVD, a covariance eigensolver, randomized SVD, and ARPACK based on the data shape. Either way you must center the data first.
- **❌ Common wrong answer:** "PCA doesn't need centering because SVD handles it."

</details>

**Q20. Should you standardize features before PCA?**

<details><summary>Show answer</summary>

- **30-second answer:** Yes when features have different units or scales: PCA maximizes variance, so a feature measured in large units (income in dollars) dominates the first component. No when all features share a unit and their scale is meaningful (e.g. pixel intensities).
- **Go deeper:** PCA on standardized data is PCA on the correlation matrix. Fit the scaler and PCA on training data only. PCA is unsupervised: it can throw away a low-variance direction that happens to be the most predictive one.
- **❌ Common wrong answer:** "Always standardize" or "never standardize" without asking about units.

</details>

### Drill 2.8 — Precision, recall, F1, and ROC-AUC from scores 🟡 (25 min)

**Problem.** Implement:
- `precision_recall_f1(y_true, y_pred)` for the positive class 1, returning 0.0 whenever a denominator is 0,
- `average_ranks(x)` — 1-based ranks where tied values share the average of their positions (like `scipy.stats.rankdata`),
- `roc_auc_from_scores(y_true, scores)` using the rank-sum (Mann–Whitney U) formula:

$$\text{AUC} = \frac{\sum_{i \in \text{pos}} r_i - n_{pos}(n_{pos}+1)/2}{n_{pos}\cdot n_{neg}}$$

**Constraints:** O(n log n) for AUC (no O(n_pos · n_neg) pair loop); ties must count as half a correct ordering.

In [24]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

gen = np.random.default_rng(4)
y_true = gen.integers(0, 2, size=300)
scores = np.round(0.8 * y_true + gen.normal(size=300), 1)          # synthetic scores, rounded → many ties
y_pred = (scores > 0.4).astype(int)
print(f"{len(np.unique(scores))} distinct scores for {len(scores)} samples → ties matter")

50 distinct scores for 300 samples → ties matter


In [25]:
def precision_recall_f1(y_true, y_pred):
    return None  # TODO: (precision, recall, f1) for class 1; 0.0 when a denominator is 0


def average_ranks(x):
    return None  # TODO: 1-based ranks; ties get the average of their positions


def roc_auc_from_scores(y_true, scores):
    return None  # TODO: ROC-AUC via the rank-sum formula


check("precision, recall, F1 vs sklearn", precision_recall_f1(y_true, y_pred),
      (precision_score(y_true, y_pred), recall_score(y_true, y_pred), f1_score(y_true, y_pred)))
check("no positive predictions → zeros, no crash", precision_recall_f1(y_true, np.zeros_like(y_true)), (0.0, 0.0, 0.0),
      hint="Guard every division: tp + fp can be 0.")
check("average ranks vs scipy.stats.rankdata", average_ranks(scores), scipy.stats.rankdata(scores),
      hint="argsort once; a tied block at 0-based sorted positions first..first+c−1 gets rank first + (c + 1) / 2.")
check("ROC-AUC vs sklearn (with ties)", roc_auc_from_scores(y_true, scores), roc_auc_score(y_true, scores),
      hint="Sum the ranks of the positives, subtract n_pos(n_pos+1)/2, divide by n_pos·n_neg.")

⏳ precision, recall, F1 vs sklearn: not attempted yet — replace None with your answer.
⏳ no positive predictions → zeros, no crash: not attempted yet — replace None with your answer.
⏳ average ranks vs scipy.stats.rankdata: not attempted yet — replace None with your answer.
⏳ ROC-AUC vs sklearn (with ties): not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def precision_recall_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return float(precision), float(recall), float(f1)


def average_ranks(x):
    x = np.asarray(x)
    order = np.argsort(x, kind="stable")
    _, first, counts = np.unique(x[order], return_index=True, return_counts=True)
    ranks = np.empty(len(x))
    ranks[order] = np.repeat(first + (counts + 1) / 2, counts)    # average 1-based rank of each tied block
    return ranks


def roc_auc_from_scores(y_true, scores):
    ranks = average_ranks(scores)
    n_pos = int(np.sum(y_true == 1))
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        raise ValueError("ROC-AUC is undefined when only one class is present")
    return float((ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


check("precision, recall, F1 vs sklearn", precision_recall_f1(y_true, y_pred),
      (precision_score(y_true, y_pred), recall_score(y_true, y_pred), f1_score(y_true, y_pred)))
check("no positive predictions → zeros, no crash", precision_recall_f1(y_true, np.zeros_like(y_true)), (0.0, 0.0, 0.0))
check("average ranks vs scipy.stats.rankdata", average_ranks(scores), scipy.stats.rankdata(scores))
check("ROC-AUC vs sklearn (with ties)", roc_auc_from_scores(y_true, scores), roc_auc_score(y_true, scores))
```

**Why the formula works:** the sum of positive ranks minus its minimum possible value, $n_{pos}(n_{pos}+1)/2$, counts how many (positive, negative) pairs are ordered correctly (ties count ½). Dividing by the number of pairs gives the probability that a random positive outscores a random negative.

**Complexity:** O(n) for the confusion counts; O(n log n) time and O(n) memory for AUC.
</details>

#### 🎤 Follow-up questions

**Q21. What does ROC-AUC measure, and when is it misleading?**

<details><summary>Show answer</summary>

- **30-second answer:** ROC-AUC is the probability that a randomly chosen positive gets a higher score than a randomly chosen negative (ties count ½) — ranking quality across all thresholds. With heavy class imbalance it can look excellent while precision at any useful threshold is poor, because the huge number of negatives keeps the false-positive *rate* small. Use PR-AUC (average precision) or precision@k when positives are rare and the top of the ranking matters.
- **Go deeper:** AUC ignores calibration: any strictly increasing transform of the scores leaves it unchanged. A random classifier has AUC 0.5 regardless of prevalence, while the PR-AUC of a random classifier equals the positive rate.
- **❌ Common wrong answer:** "AUC is the accuracy at the 0.5 threshold."

</details>

**Q22. Why report F1 instead of accuracy, and how do you choose the decision threshold?**

<details><summary>Show answer</summary>

- **30-second answer:** With 1% positives, predicting "negative" for everyone scores 99% accuracy and catches nothing. F1, the harmonic mean of precision and recall, is high only when both are. Choose the threshold on a **validation** set from the business cost of false positives vs false negatives (or a required recall), not by defaulting to 0.5 and never on the test set.
- **Go deeper:** F-β weights recall β times as much as precision. For multiclass problems, macro-F1 treats classes equally, micro-F1 pools all decisions (it equals accuracy for single-label problems), and weighted-F1 averages by class support.
- **❌ Common wrong answer:** "0.5 is the correct threshold because the model outputs probabilities."

</details>

### Drill 2.9 — Stratified k-fold indices 🟡 (15 min)

**Problem.** Implement `stratified_kfold(y, n_splits, rng)` returning a list of `(train_idx, test_idx)` pairs such that every sample is in exactly one test fold and each class is spread as evenly as possible across folds.

**Constraints:** shuffle with the given `rng`; per class, the counts across test folds may differ by at most 1.

In [26]:
from sklearn.model_selection import StratifiedKFold

y_imbalanced = np.array([0] * 50 + [1] * 12 + [2] * 5)            # imbalanced labels: 50 / 12 / 5
sk_fold_counts = np.array([np.bincount(y_imbalanced[te], minlength=3)
                           for _, te in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(y_imbalanced)), y_imbalanced)])
print("sklearn StratifiedKFold class counts per test fold:\n", sk_fold_counts)

sklearn StratifiedKFold class counts per test fold:
 [[10  3  1]
 [10  3  1]
 [10  2  1]
 [10  2  1]
 [10  2  1]]


In [27]:
def stratified_kfold(y, n_splits, rng):
    return None  # TODO: list of (train_idx, test_idx) pairs


folds = stratified_kfold(y_imbalanced, 5, np.random.default_rng(0))
if folds is None:
    check("stratified k-fold", None, True)
else:
    test_sets = [te for _, te in folds]
    check("test folds partition all indices", np.sort(np.concatenate(test_sets)), np.arange(len(y_imbalanced)))
    check("train/test disjoint and complete",
          all(len(np.intersect1d(tr, te)) == 0 and len(tr) + len(te) == len(y_imbalanced) for tr, te in folds), True)
    fold_counts = np.array([np.bincount(y_imbalanced[te], minlength=3) for te in test_sets])
    check("every class spread evenly (max − min ≤ 1)", bool((fold_counts.max(axis=0) - fold_counts.min(axis=0) <= 1).all()), True,
          hint="Shuffle within each class, then deal indices out round-robin like playing cards.")
    print("our class counts per test fold:\n", fold_counts)

⏳ stratified k-fold: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def stratified_kfold(y, n_splits, rng):
    y = np.asarray(y)
    # shuffle inside each class, then line the classes up one after another
    ordered = np.concatenate([rng.permutation(np.flatnonzero(y == c)) for c in np.unique(y)])
    fold_of = np.empty(len(y), dtype=int)
    fold_of[ordered] = np.arange(len(y)) % n_splits               # deal out like playing cards
    all_idx = np.arange(len(y))
    return [(all_idx[fold_of != f], all_idx[fold_of == f]) for f in range(n_splits)]


folds = stratified_kfold(y_imbalanced, 5, np.random.default_rng(0))
test_sets = [te for _, te in folds]
check("test folds partition all indices", np.sort(np.concatenate(test_sets)), np.arange(len(y_imbalanced)))
check("train/test disjoint and complete",
      all(len(np.intersect1d(tr, te)) == 0 and len(tr) + len(te) == len(y_imbalanced) for tr, te in folds), True)
fold_counts = np.array([np.bincount(y_imbalanced[te], minlength=3) for te in test_sets])
check("every class spread evenly (max − min ≤ 1)", bool((fold_counts.max(axis=0) - fold_counts.min(axis=0) <= 1).all()), True)
print("our class counts per test fold:\n", fold_counts)
```

Dealing continues across classes (the 2nd class starts where the 1st stopped), so total fold sizes are also balanced.

**Complexity:** O(n) to build the assignment, O(n·k) to materialize the k index pairs.
</details>

#### 🎤 Follow-up question

**Q23. When is stratified k-fold the wrong cross-validation scheme?**

<details><summary>Show answer</summary>

- **30-second answer:** Whenever rows are not independent. Several rows per patient/user/session → `GroupKFold` or `StratifiedGroupKFold`, so one person never appears in both train and test. Time series → `TimeSeriesSplit` (always train on the past, validate on the future). Tuning hyperparameters *and* reporting performance → nested CV or a held-out test set.
- **Go deeper:** More folds means each model trains on more data (less pessimistic estimate) at higher compute cost; the fold scores are correlated, so their standard deviation understates uncertainty. For small datasets, repeated stratified k-fold stabilizes the estimate.
- **❌ Common wrong answer:** "Shuffle and stratify — that works for every dataset."

</details>

## 3. Deep Learning Building Blocks 🔴

Research-engineer and ML-engineer loops often ask you to implement a layer's forward (and sometimes backward) pass in NumPy or raw PyTorch tensors. The checkers compare against PyTorch in `float64`, so a match really means "same math".

> 💡 Review: [Neural Networks From Scratch](../04_Deep_Learning/01_Neural_Networks_From_Scratch.ipynb) · [PyTorch](../04_Deep_Learning/02_PyTorch.ipynb) · [Transformers From Scratch](../04_Deep_Learning/03_Transformers_From_Scratch.ipynb)

### Drill 3.1 — MLP forward + backward with a gradient check 🔴 (35 min)

**Problem.** For a 2-layer network `X → Linear(W1, b1) → ReLU → Linear(W2, b2) → softmax cross-entropy (mean)`, implement `mlp_loss_and_grads(params, X, y)` returning `(loss, grads)`, where `grads` has the same keys and shapes as `params`. Then implement `numerical_gradient(f, x, eps)` with central differences and use it to verify your backward pass.

**Shapes:** `X (n, d)`, `W1 (d, h)`, `b1 (h,)`, `W2 (h, C)`, `b2 (C,)` — layers compute `X @ W1 + b1`.

**Constraints:** vectorized over the batch; relative error between analytic and numerical gradients below 1e-6.

In [28]:
gen = np.random.default_rng(5)
X_mlp = gen.normal(size=(8, 5))
y_mlp = gen.integers(0, 3, size=8)
params = {"W1": gen.normal(size=(5, 6)) * 0.5, "b1": gen.normal(size=6) * 0.1,
          "W2": gen.normal(size=(6, 3)) * 0.5, "b2": np.zeros(3)}

# reference: the same network in PyTorch (float64), differentiated by autograd
t_params = {name: torch.tensor(value, requires_grad=True) for name, value in params.items()}
t_hidden = torch.relu(torch.tensor(X_mlp) @ t_params["W1"] + t_params["b1"])
t_mlp_loss = F.cross_entropy(t_hidden @ t_params["W2"] + t_params["b2"], torch.tensor(y_mlp))
t_mlp_loss.backward()
print(f"PyTorch loss: {t_mlp_loss.item():.6f}")

PyTorch loss: 1.180505


In [29]:
def mlp_loss_and_grads(params, X, y):
    return None  # TODO: return (loss, grads) with grads[name].shape == params[name].shape


def numerical_gradient(f, x, eps=1e-6):
    """Central differences: perturb each entry of array x IN PLACE, call f() for the loss, restore the entry."""
    return None  # TODO


mlp_out = mlp_loss_and_grads(params, X_mlp, y_mlp)
check("loss vs torch", None if mlp_out is None else mlp_out[0], t_mlp_loss.item())
for name in params:
    check(f"d{name} vs torch autograd", None if mlp_out is None else mlp_out[1][name], t_params[name].grad.numpy(),
          hint="dlogits = (softmax − one_hot)/n; dW2 = hᵀ·dlogits; dh = dlogits·W2ᵀ; dz1 = dh·(z1 > 0); dW1 = Xᵀ·dz1.")
num_grad = None if mlp_out is None else numerical_gradient(lambda: mlp_loss_and_grads(params, X_mlp, y_mlp)[0], params["W1"])
if num_grad is None:
    check("gradient check on W1", None, True)
else:
    rel_error = np.abs(num_grad - mlp_out[1]["W1"]).max() / max(1e-12, np.abs(num_grad).max() + np.abs(mlp_out[1]["W1"]).max())
    print(f"relative error = {rel_error:.2e}")
    check("gradient check on W1 (relative error < 1e-6)", bool(rel_error < 1e-6), True)

⏳ loss vs torch: not attempted yet — replace None with your answer.
⏳ dW1 vs torch autograd: not attempted yet — replace None with your answer.
⏳ db1 vs torch autograd: not attempted yet — replace None with your answer.
⏳ dW2 vs torch autograd: not attempted yet — replace None with your answer.
⏳ db2 vs torch autograd: not attempted yet — replace None with your answer.
⏳ gradient check on W1: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def mlp_loss_and_grads(params, X, y):
    W1, b1, W2, b2 = params["W1"], params["b1"], params["W2"], params["b2"]
    n = len(X)
    # forward — keep what backward needs
    z1 = X @ W1 + b1
    h = np.maximum(z1, 0.0)
    logits = h @ W2 + b2
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    loss = -log_probs[np.arange(n), y].mean()
    # backward — chain rule, layer by layer
    dlogits = np.exp(log_probs)
    dlogits[np.arange(n), y] -= 1.0
    dlogits /= n
    grads = {"W2": h.T @ dlogits, "b2": dlogits.sum(axis=0)}
    dz1 = (dlogits @ W2.T) * (z1 > 0)                   # ReLU passes gradient only where it was active
    grads["W1"] = X.T @ dz1
    grads["b1"] = dz1.sum(axis=0)
    return float(loss), grads


def numerical_gradient(f, x, eps=1e-6):
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        original = x[idx]
        x[idx] = original + eps
        loss_plus = f()
        x[idx] = original - eps
        loss_minus = f()
        x[idx] = original                               # always restore
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
    return grad


mlp_out = mlp_loss_and_grads(params, X_mlp, y_mlp)
check("loss vs torch", mlp_out[0], t_mlp_loss.item())
for name in params:
    check(f"d{name} vs torch autograd", mlp_out[1][name], t_params[name].grad.numpy())
num_grad = numerical_gradient(lambda: mlp_loss_and_grads(params, X_mlp, y_mlp)[0], params["W1"])
rel_error = np.abs(num_grad - mlp_out[1]["W1"]).max() / max(1e-12, np.abs(num_grad).max() + np.abs(mlp_out[1]["W1"]).max())
print(f"relative error = {rel_error:.2e}")
check("gradient check on W1 (relative error < 1e-6)", bool(rel_error < 1e-6), True)
```

**Complexity:** forward and backward are both O(n·(d·h + h·C)) time; memory O(n·(h + C)) for cached activations. The numerical gradient needs 2 forward passes **per parameter** — fine for testing, useless for training.
</details>

#### 🎤 Follow-up questions

**Q24. How do you run a gradient check correctly?**

<details><summary>Show answer</summary>

- **30-second answer:** Compare the analytic gradient with central differences $(f(x+\varepsilon) - f(x-\varepsilon))/2\varepsilon$ on a tiny network in **float64**, using the relative error $|g_a - g_n| / (|g_a| + |g_n|)$. Around 1e-7 or smaller is healthy; above ~1e-3 means a bug.
- **Go deeper:** Central differences have O(ε²) truncation error vs O(ε) for one-sided differences, but an ε that's too small makes round-off dominate — ε ≈ 1e-5 to 1e-6 works in float64. Disable dropout and other randomness, watch for kinks (ReLU exactly at 0), include the same regularization terms in both, and for large models check a random subset of coordinates.
- **❌ Common wrong answer:** "Use float32 and ε = 1e-10 for maximum precision" — round-off error swamps the difference.

</details>

**Q25. How expensive is the backward pass compared with the forward pass, and why does training use so much memory?**

<details><summary>Show answer</summary>

- **30-second answer:** Backward costs roughly twice the forward compute (gradients w.r.t. both activations and weights), so a training step is about 3× a forward pass. Memory grows because every layer's activations are stored for the backward pass — proportional to batch size × depth × width (× sequence length for transformers) — on top of weights, gradients, and optimizer state.
- **Go deeper:** That's the origin of the common estimate "training compute ≈ 6 × parameters × tokens" (2N per token forward, 4N backward). Activation checkpointing recomputes activations during backward to trade ~one extra forward pass for much less memory. Reverse-mode autodiff gets *all* parameter gradients in one backward pass because the loss is a single scalar.
- **❌ Common wrong answer:** "Backward is as cheap as forward and needs no extra memory."

</details>

### Drill 3.2 — conv2d forward pass with im2col 🔴 (30 min)

**Problem.** Implement `conv2d_forward(X, W, b, stride, padding)` for `X (N, C, H, W_in)`, filters `W (F, C, kh, kw)`, bias `b (F,)`, returning `(N, F, H_out, W_out)` with

$$H_{out} = \left\lfloor \frac{H + 2p - k_h}{s} \right\rfloor + 1$$

**Constraints:** no Python loops over pixels — extract all patches ("im2col"), then one matrix multiply. Like every deep-learning library, compute **cross-correlation** (don't flip the kernel).

In [30]:
gen = np.random.default_rng(6)
X_img = gen.normal(size=(2, 3, 7, 6))
W_conv = gen.normal(size=(4, 3, 3, 2))
b_conv = gen.normal(size=4)
conv_cases = [(1, 0), (2, 1), (3, 2)]
conv_refs = {case: F.conv2d(torch.tensor(X_img), torch.tensor(W_conv), torch.tensor(b_conv), stride=case[0], padding=case[1]).numpy()
             for case in conv_cases}
print({case: ref.shape for case, ref in conv_refs.items()})

{(1, 0): (2, 4, 5, 5), (2, 1): (2, 4, 4, 4), (3, 2): (2, 4, 3, 3)}


In [31]:
def conv2d_forward(X, W, b, stride=1, padding=0):
    return None  # TODO: (N, F, H_out, W_out), no loops over pixels


for stride, padding in conv_cases:
    check(f"conv2d stride={stride} padding={padding} vs torch", conv2d_forward(X_img, W_conv, b_conv, stride, padding),
          conv_refs[(stride, padding)],
          hint="Pad, sliding_window_view(Xp, (kh, kw), axis=(2, 3))[:, :, ::s, ::s], reshape patches to (N, H_out, W_out, C·kh·kw), @ W.reshape(F, −1).T + b.")

⏳ conv2d stride=1 padding=0 vs torch: not attempted yet — replace None with your answer.
⏳ conv2d stride=2 padding=1 vs torch: not attempted yet — replace None with your answer.
⏳ conv2d stride=3 padding=2 vs torch: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def conv2d_forward(X, W, b, stride=1, padding=0):
    N, C, H, W_in = X.shape
    n_filters, _, kh, kw = W.shape
    Xp = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)))
    H_out = (H + 2 * padding - kh) // stride + 1
    W_out = (W_in + 2 * padding - kw) // stride + 1
    # every kh×kw window as a zero-copy view: (N, C, H_out, W_out, kh, kw)
    windows = np.lib.stride_tricks.sliding_window_view(Xp, (kh, kw), axis=(2, 3))[:, :, ::stride, ::stride]
    cols = windows.transpose(0, 2, 3, 1, 4, 5).reshape(N, H_out, W_out, C * kh * kw)   # im2col (this copies)
    out = cols @ W.reshape(n_filters, -1).T + b                                        # (N, H_out, W_out, F)
    return out.transpose(0, 3, 1, 2)


for stride, padding in conv_cases:
    check(f"conv2d stride={stride} padding={padding} vs torch", conv2d_forward(X_img, W_conv, b_conv, stride, padding),
          conv_refs[(stride, padding)])
```

The flatten order matters: the patch is flattened as (C, kh, kw), which is exactly how `W.reshape(F, -1)` flattens each filter.

**Complexity:** O(N · H_out · W_out · C·kh·kw · F) time; the column matrix needs O(N · H_out · W_out · C·kh·kw) memory — roughly kh·kw times the input size.
</details>

#### 🎤 Follow-up questions

**Q26. What is the output size and the parameter count of a convolution layer?**

<details><summary>Show answer</summary>

- **30-second answer:** $H_{out} = \lfloor (H + 2p - k)/s \rfloor + 1$ (same for width; with dilation d replace k by d(k − 1) + 1). Parameters = F·C·kh·kw + F biases. Example: a 3×3 conv from 64 to 128 channels has 128·64·9 + 128 = 73,856 parameters — whatever the image size.
- **Go deeper:** Weight sharing is why convs are parameter-efficient compared with a dense layer on pixels. "Same" padding for stride 1 and odd k is p = (k − 1)/2. A 1×1 conv mixes channels only; depthwise-separable convs cut parameters roughly by a factor of k² for large channel counts.
- **❌ Common wrong answer:** "The parameter count depends on the input height and width."

</details>

**Q27. Why is im2col fast, and what does it cost?**

<details><summary>Show answer</summary>

- **30-second answer:** It turns convolution into a single large matrix multiplication, which BLAS libraries and GPUs execute extremely efficiently (cache-friendly, vectorized, multi-threaded). The price is memory: the patch matrix is about kh·kw times bigger than the input.
- **Go deeper:** Libraries choose among several algorithms — im2col + GEMM, implicit GEMM, Winograd (fast for 3×3), FFT, direct — often by benchmarking at run time (in PyTorch, `torch.backends.cudnn.benchmark = True`). The backward pass uses the transpose operation, col2im.
- **❌ Common wrong answer:** "im2col saves memory."

</details>

### Drill 3.3 — Batch normalization forward: train vs eval 🟡 (20 min)

**Problem.** Implement `batchnorm_forward(x, gamma, beta, running_mean, running_var, training, momentum=0.1, eps=1e-5)` for `x (N, D)`, returning `(out, new_running_mean, new_running_var)` with **PyTorch conventions**:
- training: normalize with the batch mean and the *biased* batch variance (÷N); update `running = (1 − momentum)·running + momentum·batch_stat`, storing the *unbiased* variance (÷(N−1)) in `running_var`;
- eval: normalize with the running statistics and leave them unchanged.

$$\text{out} = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

In [32]:
gen = np.random.default_rng(7)
x_bn = gen.normal(loc=3.0, scale=2.0, size=(16, 4))
gamma, beta = gen.normal(size=4), gen.normal(size=4)

t_running_mean, t_running_var = torch.zeros(4, dtype=torch.float64), torch.ones(4, dtype=torch.float64)
t_bn_train = F.batch_norm(torch.tensor(x_bn), t_running_mean, t_running_var, torch.tensor(gamma), torch.tensor(beta),
                          training=True, momentum=0.1, eps=1e-5).numpy()          # updates the running stats in place
t_bn_eval = F.batch_norm(torch.tensor(x_bn), t_running_mean, t_running_var, torch.tensor(gamma), torch.tensor(beta),
                         training=False, eps=1e-5).numpy()
print("PyTorch running_mean after one step:", t_running_mean.numpy().round(4))

PyTorch running_mean after one step: [0.2514 0.2795 0.241  0.2546]


In [33]:
def batchnorm_forward(x, gamma, beta, running_mean, running_var, training, momentum=0.1, eps=1e-5):
    return None  # TODO: return (out, new_running_mean, new_running_var)


bn_train = batchnorm_forward(x_bn, gamma, beta, np.zeros(4), np.ones(4), training=True)
check("train-mode output vs torch", None if bn_train is None else bn_train[0], t_bn_train,
      hint="Normalize with the BATCH mean and variance (np.var divides by N).")
check("running_mean update", None if bn_train is None else bn_train[1], t_running_mean.numpy(),
      hint="(1 − momentum)·running_mean + momentum·batch_mean")
check("running_var update", None if bn_train is None else bn_train[2], t_running_var.numpy(),
      hint="PyTorch stores the UNBIASED batch variance (× N / (N − 1)) in running_var.")
bn_eval = None if bn_train is None else batchnorm_forward(x_bn, gamma, beta, bn_train[1], bn_train[2], training=False)
check("eval-mode output vs torch", None if bn_eval is None else bn_eval[0], t_bn_eval,
      hint="In eval mode use the running statistics, never the batch statistics.")

⏳ train-mode output vs torch: not attempted yet — replace None with your answer.
⏳ running_mean update: not attempted yet — replace None with your answer.
⏳ running_var update: not attempted yet — replace None with your answer.
⏳ eval-mode output vs torch: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def batchnorm_forward(x, gamma, beta, running_mean, running_var, training, momentum=0.1, eps=1e-5):
    if training:
        n = x.shape[0]
        mu, var = x.mean(axis=0), x.var(axis=0)                     # biased variance (÷ N) for normalizing
        x_hat = (x - mu) / np.sqrt(var + eps)
        running_mean = (1 - momentum) * running_mean + momentum * mu
        running_var = (1 - momentum) * running_var + momentum * var * n / (n - 1)   # unbiased for the running estimate
    else:
        x_hat = (x - running_mean) / np.sqrt(running_var + eps)
    return gamma * x_hat + beta, running_mean, running_var


bn_train = batchnorm_forward(x_bn, gamma, beta, np.zeros(4), np.ones(4), training=True)
check("train-mode output vs torch", bn_train[0], t_bn_train)
check("running_mean update", bn_train[1], t_running_mean.numpy())
check("running_var update", bn_train[2], t_running_var.numpy())
bn_eval = batchnorm_forward(x_bn, gamma, beta, bn_train[1], bn_train[2], training=False)
check("eval-mode output vs torch", bn_eval[0], t_bn_eval)
```

For images `(N, C, H, W)` the statistics are per channel: take the mean over axes `(0, 2, 3)`.

**Complexity:** O(N·D) time, O(N·D) memory for the output (plus the cached `x_hat` and variance if you implement backward).
</details>

#### 🎤 Follow-up questions

**Q28. Why does BatchNorm behave differently in training and evaluation, and what goes wrong if you forget `model.eval()`?**

<details><summary>Show answer</summary>

- **30-second answer:** In training it normalizes with the current batch's statistics (which also adds helpful noise); at inference the output for one example must not depend on the other examples in the batch, and batch size may be 1, so it uses running averages collected during training. Forgetting `model.eval()` makes predictions depend on batch composition and corrupts the running statistics with test data.
- **Go deeper:** Small or non-i.i.d. batches make batch statistics noisy — freeze BN statistics when fine-tuning with tiny batches, or use GroupNorm/LayerNorm. Conventions differ: PyTorch `momentum=0.1` is the weight on the *new* batch, while Keras `momentum=0.99` is the weight on the *old* running value.
- **❌ Common wrong answer:** "BatchNorm is the same function at train and test time, it just has learnable γ and β."

</details>

**Q29. BatchNorm vs LayerNorm — why do transformers use LayerNorm?**

<details><summary>Show answer</summary>

- **30-second answer:** BatchNorm normalizes each feature across the batch; LayerNorm normalizes each example across its features. LayerNorm doesn't depend on batch size or on other sequences, behaves identically in training and inference, and handles variable-length, padded sequences and token-by-token generation — all awkward for BatchNorm.
- **Go deeper:** Most modern transformers put the norm *before* each sub-layer (pre-LN), which trains more stably than the original post-LN; many LLMs (e.g. the Llama family) use RMSNorm, which drops the mean subtraction and the bias.
- **❌ Common wrong answer:** "Transformers use LayerNorm because it's faster than BatchNorm."

</details>

### Drill 3.4 — Inverted dropout 🟢 (10 min)

**Problem.** Implement `dropout(x, p, rng, training)`: during training, zero each entry independently with probability `p` and scale survivors by `1/(1 − p)`; during evaluation return `x` unchanged.

**Why these checks:** dropout is random, so we test its *properties* — the zero fraction, the survivor value, the preserved mean, and the eval identity.

In [34]:
def dropout(x, p, rng, training=True):
    return None  # TODO: inverted dropout


x_ones = np.full((1000, 100), 2.0)
p_drop = 0.3
dropped = dropout(x_ones, p_drop, np.random.default_rng(8), training=True)
check("eval mode returns x unchanged", dropout(x_ones, p_drop, np.random.default_rng(8), training=False), x_ones)
check("≈30% of entries zeroed (±1%)", None if dropped is None else bool(abs((dropped == 0).mean() - p_drop) < 0.01), True,
      hint="mask = rng.random(x.shape) >= p")
check("survivors scaled by 1/(1 − p)", None if dropped is None else np.unique(dropped[dropped != 0]), [2.0 / (1 - p_drop)],
      hint="Multiply by the mask AND divide by (1 − p).")
check("mean preserved (±1%)", None if dropped is None else bool(abs(dropped.mean() - 2.0) < 0.02), True)
t_dropped = F.dropout(torch.ones(10_000), p=p_drop, training=True)
print("PyTorch survivors are scaled the same way:", torch.unique(t_dropped[t_dropped != 0]).tolist(), "| 1/(1 − p) =", 1 / (1 - p_drop))

⏳ eval mode returns x unchanged: not attempted yet — replace None with your answer.
⏳ ≈30% of entries zeroed (±1%): not attempted yet — replace None with your answer.
⏳ survivors scaled by 1/(1 − p): not attempted yet — replace None with your answer.
⏳ mean preserved (±1%): not attempted yet — replace None with your answer.
PyTorch survivors are scaled the same way: [1.4285714626312256] | 1/(1 − p) = 1.4285714285714286


<details><summary>💡 Show solution</summary>

```python
def dropout(x, p, rng, training=True):
    if not training or p == 0.0:
        return x
    keep = rng.random(x.shape) >= p                     # True with probability 1 − p
    return x * keep / (1.0 - p)                         # scale now, so inference needs no change


x_ones = np.full((1000, 100), 2.0)
p_drop = 0.3
dropped = dropout(x_ones, p_drop, np.random.default_rng(8), training=True)
check("eval mode returns x unchanged", dropout(x_ones, p_drop, np.random.default_rng(8), training=False), x_ones)
check("≈30% of entries zeroed (±1%)", bool(abs((dropped == 0).mean() - p_drop) < 0.01), True)
check("survivors scaled by 1/(1 − p)", np.unique(dropped[dropped != 0]), [2.0 / (1 - p_drop)])
check("mean preserved (±1%)", bool(abs(dropped.mean() - 2.0) < 0.02), True)
print(f"zero fraction {(dropped == 0).mean():.4f} | mean {dropped.mean():.4f}")
```

Handle `p = 1` explicitly in production code (it would divide by zero); frameworks return zeros.

**Complexity:** O(n) time and O(n) memory for the mask (which backward reuses).
</details>

#### 🎤 Follow-up questions

**Q30. Why "inverted" dropout — why scale during training instead of at test time?**

<details><summary>Show answer</summary>

- **30-second answer:** Dividing survivors by (1 − p) keeps each activation's expected value equal to its input during training, so inference is simply the identity: no scaling code at test time, and changing p never touches the serving path. The original formulation instead multiplied weights by (1 − p) at test time — equivalent in expectation.
- **Go deeper:** Dropout can be viewed as training an ensemble of thinned sub-networks with shared weights; the test-time network approximates averaging them. Its variance shift can interact badly with BatchNorm placed right after it.
- **❌ Common wrong answer:** "Dropout is also applied at inference to keep regularizing the model."

</details>

**Q31. What is Monte Carlo dropout, and what's the trap when enabling it in PyTorch?**

<details><summary>Show answer</summary>

- **30-second answer:** Keep dropout active at inference, run T stochastic forward passes, and use the mean as the prediction and the spread (variance or predictive entropy) as an uncertainty estimate; Gal & Ghahramani (2016) interpret this as approximate Bayesian inference. It costs T× compute and its uncertainty estimates can still be miscalibrated.
- **Go deeper:** The trap: calling `model.train()` also switches BatchNorm to batch statistics and running-stat updates. Instead call `model.eval()` and then set only the dropout modules to training mode (`m.train()` for `nn.Dropout` modules).
- **❌ Common wrong answer:** "Just call `model.train()` before predicting."

</details>

### Drill 3.5 — An Adam (and AdamW) update step 🟡 (20 min)

**Problem.** Implement `adam_step(param, grad, m, v, t, lr, beta1, beta2, eps)` returning `(new_param, new_m, new_v)`, with `t` starting at 1:

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \quad v \leftarrow \beta_2 v + (1-\beta_2) g^2, \quad \hat m = \frac{m}{1-\beta_1^t}, \quad \hat v = \frac{v}{1-\beta_2^t}, \quad \theta \leftarrow \theta - \text{lr}\,\frac{\hat m}{\sqrt{\hat v} + \varepsilon}$$

Then implement `adamw_step(…, weight_decay)`: first shrink the weights, $\theta \leftarrow \theta\,(1 - \text{lr}\cdot\lambda)$, then do the Adam update. The checker runs 6 steps and compares with `torch.optim.Adam` / `AdamW`.

In [35]:
gen = np.random.default_rng(9)
adam_start = gen.normal(size=5)
grad_sequence = gen.normal(size=(6, 5))                       # pretend gradients for 6 steps


def run_numpy_optimizer(step_fn, **kwargs):
    param, m, v = adam_start.copy(), np.zeros(5), np.zeros(5)
    for t, g in enumerate(grad_sequence, start=1):
        out = step_fn(param, g, m, v, t, lr=0.1, **kwargs)
        if out is None:
            return None
        param, m, v = out
    return param


def run_torch_optimizer(optimizer_class, **kwargs):
    p = torch.tensor(adam_start, requires_grad=True)
    optimizer = optimizer_class([p], lr=0.1, **kwargs)
    for g in grad_sequence:
        p.grad = torch.tensor(g)
        optimizer.step()
    return p.detach().numpy()


torch_adam, torch_adamw = run_torch_optimizer(torch.optim.Adam), run_torch_optimizer(torch.optim.AdamW, weight_decay=0.1)
print("torch Adam after 6 steps :", torch_adam.round(4))
print("torch AdamW after 6 steps:", torch_adamw.round(4))

torch Adam after 6 steps : [-0.4062 -0.0897 -1.8015  0.4915  1.1739]
torch AdamW after 6 steps: [-0.371  -0.093  -1.6959  0.4536  1.1046]


In [36]:
def adam_step(param, grad, m, v, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
    return None  # TODO: return (new_param, new_m, new_v)


def adamw_step(param, grad, m, v, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
    return None  # TODO: decoupled weight decay, then the Adam update


check("Adam: 6 steps vs torch.optim.Adam", run_numpy_optimizer(adam_step), torch_adam,
      hint="Update m and v, bias-correct with 1 − β^t, then param − lr · m̂ / (√v̂ + eps).")
check("AdamW: 6 steps vs torch.optim.AdamW", run_numpy_optimizer(adamw_step, weight_decay=0.1), torch_adamw,
      hint="param ← param · (1 − lr · weight_decay) BEFORE the Adam update; the decay never touches m or v.")

⏳ Adam: 6 steps vs torch.optim.Adam: not attempted yet — replace None with your answer.
⏳ AdamW: 6 steps vs torch.optim.AdamW: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def adam_step(param, grad, m, v, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad                  # EMA of gradients (direction, momentum)
    v = beta2 * v + (1 - beta2) * grad ** 2             # EMA of squared gradients (per-parameter scale)
    m_hat = m / (1 - beta1 ** t)                        # undo the bias toward the zero initialization
    v_hat = v / (1 - beta2 ** t)
    return param - lr * m_hat / (np.sqrt(v_hat) + eps), m, v


def adamw_step(param, grad, m, v, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
    param = param * (1 - lr * weight_decay)             # decoupled decay: not rescaled by √v̂
    return adam_step(param, grad, m, v, t, lr, beta1, beta2, eps)


check("Adam: 6 steps vs torch.optim.Adam", run_numpy_optimizer(adam_step), torch_adam)
check("AdamW: 6 steps vs torch.optim.AdamW", run_numpy_optimizer(adamw_step, weight_decay=0.1), torch_adamw)

g = 0.5                                                 # first step with default betas and a constant gradient
with_correction = (0.1 * g / (1 - 0.9)) / np.sqrt(0.001 * g ** 2 / (1 - 0.999))
without_correction = (0.1 * g) / np.sqrt(0.001 * g ** 2)
print(f"step-1 update size in units of lr: with bias correction {with_correction:.3f}, without {without_correction:.3f}")
```

**Complexity:** O(P) time per step for P parameters; Adam stores two extra tensors (m, v), so optimizer state is 2× the parameter memory.
</details>

#### 🎤 Follow-up questions

**Q32. What do Adam's two moments do, and why does it need bias correction?**

<details><summary>Show answer</summary>

- **30-second answer:** m is an exponential moving average of gradients (momentum: a smoothed direction); v is an EMA of squared gradients (a per-parameter scale). Dividing m̂ by √v̂ gives each parameter an update of roughly `lr` in size regardless of its gradient's scale. Both start at 0, so early averages are biased toward 0; without correction the first step with default betas is about 3.16 × lr instead of 1 × lr (computed in the solution).
- **Go deeper:** With a constant gradient g, $m_t = (1-\beta_1^t)\,g$ — dividing by $1-\beta_1^t$ recovers g exactly. ε prevents division by zero and caps the step for parameters with tiny gradients. A learning-rate warm-up is often used on top, because v̂ is still noisy in the first steps.
- **❌ Common wrong answer:** "Bias correction is a form of regularization."

</details>

**Q33. What's the difference between Adam with L2 regularization and AdamW?**

<details><summary>Show answer</summary>

- **30-second answer:** Adam + L2 adds λθ to the gradient, which then gets divided by √v̂ — so parameters with large historical gradients are regularized *less*. AdamW decouples weight decay: it shrinks the weights directly, θ ← θ(1 − lr·λ), independent of the adaptive scaling. Loshchilov & Hutter showed this generalizes better; AdamW is the default for training transformers.
- **Go deeper:** Typical weight decay is 0.01–0.1, usually not applied to biases and normalization gains. In PyTorch, `torch.optim.Adam(weight_decay=…)` is the coupled L2 form by default, while `torch.optim.AdamW` is decoupled — the checker above confirms our decoupled update matches it.
- **❌ Common wrong answer:** "They're identical; AdamW is just Adam with the weight_decay argument set."

</details>

### Drill 3.6 — Scaled dot-product and multi-head attention with a causal mask 🔴 (35 min)

**Problem.** Implement:
- `causal_mask(T)` → `(T, T)` boolean, **True where attention is allowed** (on and below the diagonal),
- `scaled_dot_product_attention(Q, K, V, mask)` → `(output, weights)` for `Q, K (…, T, d_k)`, `V (…, T, d_v)`:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V, \qquad M_{ij} = 0 \text{ if allowed else } -\infty$$

- `multi_head_attention(X, W_q, W_k, W_v, W_o, n_heads, causal)` for `X (B, T, D)`: project (`X @ W.T`, no biases), split into heads, attend per head, concatenate, project with `W_o`.

**Watch the convention:** our mask uses True = *allowed*; PyTorch's `nn.MultiheadAttention` boolean `attn_mask` uses True = *not allowed*. Mixing these up is a classic bug.

In [37]:
gen = np.random.default_rng(10)
Q_att, K_att, V_att = gen.normal(size=(2, 3, 5, 4)), gen.normal(size=(2, 3, 5, 4)), gen.normal(size=(2, 3, 5, 6))
ref_sdpa = F.scaled_dot_product_attention(torch.tensor(Q_att), torch.tensor(K_att), torch.tensor(V_att), is_causal=True).numpy()

B_att, T_att, D_att, H_att = 2, 5, 8, 2
X_att = gen.normal(size=(B_att, T_att, D_att))
W_q, W_k, W_v, W_o = (gen.normal(size=(D_att, D_att)) * 0.3 for _ in range(4))
torch_mha = torch.nn.MultiheadAttention(D_att, H_att, bias=False, batch_first=True, dtype=torch.float64)
with torch.no_grad():
    torch_mha.in_proj_weight.copy_(torch.tensor(np.concatenate([W_q, W_k, W_v])))      # PyTorch stacks [W_q, W_k, W_v]
    torch_mha.out_proj.weight.copy_(torch.tensor(W_o))
    x_t = torch.tensor(X_att)
    torch_blocked = torch.triu(torch.ones(T_att, T_att, dtype=torch.bool), diagonal=1)  # PyTorch: True = NOT allowed
    ref_mha = torch_mha(x_t, x_t, x_t, attn_mask=torch_blocked, need_weights=False)[0].numpy()
print("reference shapes:", ref_sdpa.shape, ref_mha.shape)

reference shapes: (2, 3, 5, 6) (2, 5, 8)


In [38]:
def causal_mask(T):
    return None  # TODO: (T, T) bool, True on and below the diagonal


def scaled_dot_product_attention(Q, K, V, mask=None):
    return None  # TODO: return (output, weights); mask True = may attend


def multi_head_attention(X, W_q, W_k, W_v, W_o, n_heads, causal=True):
    return None  # TODO: (B, T, D) output


check("causal_mask(3)", causal_mask(3), [[True, False, False], [True, True, False], [True, True, True]],
      hint="np.tril(np.ones((T, T), dtype=bool))")
mask5 = causal_mask(5)
sdpa_out = None if mask5 is None else scaled_dot_product_attention(Q_att, K_att, V_att, mask5)
check("SDPA vs torch (is_causal=True)", None if sdpa_out is None else sdpa_out[0], ref_sdpa,
      hint="scores = Q @ K.swapaxes(-1, -2) / sqrt(d_k); np.where(mask, scores, -np.inf); stable softmax; weights @ V.")
check("weights: rows sum to 1 and the future gets 0",
      None if sdpa_out is None else bool(np.allclose(sdpa_out[1].sum(axis=-1), 1) and np.all(sdpa_out[1][..., ~mask5] == 0)), True)
check("multi-head attention vs nn.MultiheadAttention", multi_head_attention(X_att, W_q, W_k, W_v, W_o, H_att), ref_mha,
      hint="(B, T, D) → (B, T, h, D/h) → transpose to (B, h, T, D/h) → attend → transpose back → reshape → @ W_o.T")

⏳ causal_mask(3): not attempted yet — replace None with your answer.
⏳ SDPA vs torch (is_causal=True): not attempted yet — replace None with your answer.
⏳ weights: rows sum to 1 and the future gets 0: not attempted yet — replace None with your answer.
⏳ multi-head attention vs nn.MultiheadAttention: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def causal_mask(T):
    return np.tril(np.ones((T, T), dtype=bool))


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)            # (..., T, T)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)                     # blocked positions get zero weight after softmax
    shifted = scores - scores.max(axis=-1, keepdims=True)            # stable softmax (the max is finite: the diagonal is allowed)
    weights = np.exp(shifted)
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights @ V, weights


def multi_head_attention(X, W_q, W_k, W_v, W_o, n_heads, causal=True):
    B, T, D = X.shape
    head_dim = D // n_heads

    def split_heads(M):                                              # (B, T, D) → (B, h, T, D/h)
        return M.reshape(B, T, n_heads, head_dim).transpose(0, 2, 1, 3)

    Q, K, V = split_heads(X @ W_q.T), split_heads(X @ W_k.T), split_heads(X @ W_v.T)
    out, _ = scaled_dot_product_attention(Q, K, V, causal_mask(T) if causal else None)
    return out.transpose(0, 2, 1, 3).reshape(B, T, D) @ W_o.T        # concatenate heads, then output projection


check("causal_mask(3)", causal_mask(3), [[True, False, False], [True, True, False], [True, True, True]])
mask5 = causal_mask(5)
sdpa_out = scaled_dot_product_attention(Q_att, K_att, V_att, mask5)
check("SDPA vs torch (is_causal=True)", sdpa_out[0], ref_sdpa)
check("weights: rows sum to 1 and the future gets 0",
      bool(np.allclose(sdpa_out[1].sum(axis=-1), 1) and np.all(sdpa_out[1][..., ~mask5] == 0)), True)
check("multi-head attention vs nn.MultiheadAttention", multi_head_attention(X_att, W_q, W_k, W_v, W_o, H_att), ref_mha)
```

**Complexity:** O(B · h · T² · d_k) time for the score matrices plus O(B·T·D²) for the projections; O(B · h · T²) memory for the attention weights — the T² term is what limits long contexts.
</details>

#### 🎤 Follow-up questions

**Q34. Why divide by √d_k in scaled dot-product attention?**

<details><summary>Show answer</summary>

- **30-second answer:** If query and key components are independent with mean 0 and variance 1, their dot product has variance d_k, so for large d_k the scores are large and softmax saturates into a near one-hot distribution with tiny gradients. Dividing by √d_k brings the variance back to about 1. (The theory notebook demonstrates this numerically.)
- **Go deeper:** Some architectures add normalization of queries and keys (QK-norm) or a learned temperature to control attention-logit growth. Masking must happen *before* the softmax (add −∞, or the dtype's minimum value in float16), not by zeroing weights afterwards, which would break the normalization.
- **❌ Common wrong answer:** "To normalize the output vectors to unit length" or "to make it faster."

</details>

**Q35. What is the complexity of self-attention, and how do FlashAttention and the KV cache help?**

<details><summary>Show answer</summary>

- **30-second answer:** Self-attention is O(T²·d) time and O(T²) memory per layer for sequence length T. FlashAttention computes the *exact* same result in blocks without materializing the T × T matrix, cutting memory to O(T) and running faster by reducing slow GPU memory reads/writes — but time is still quadratic. During generation, a KV cache stores past keys and values so each new token costs O(T) instead of recomputing the whole prefix.
- **Go deeper:** Approximate alternatives — sliding-window/local, sparse, and linear attention — reduce the quadratic cost at some quality risk. Multi-query and grouped-query attention (MQA/GQA) share key/value heads to shrink the KV cache, which dominates memory at long context.
- **❌ Common wrong answer:** "FlashAttention makes attention linear time."

</details>

### Drill 3.7 — Temperature, top-k, and top-p sampling 🟡 (20 min)

**Problem.** Implement `sampling_probs(logits, temperature, top_k, top_p)` returning the renormalized probability vector the sampler would draw from, applying the steps **in this order** (as Hugging Face `generate` does): divide logits by temperature → keep only the `top_k` largest logits → keep the smallest set of most-likely tokens whose cumulative probability reaches `top_p`. Then implement `sample_token(logits, rng, **kwargs)`.

**Top-p rule:** sort by probability; keep a token if the total probability of the tokens ranked **above** it is still < p.

**Example:** probabilities `[0.5, 0.3, 0.15, 0.05]` with `top_p = 0.75` keep the first two tokens → `[0.625, 0.375, 0, 0]`.

In [39]:
from transformers.generation.logits_process import TemperatureLogitsWarper, TopKLogitsWarper, TopPLogitsWarper

gen = np.random.default_rng(11)
hf_cases = []
for temperature, top_k, top_p in [(1.0, 5, 0.8), (0.7, None, 0.9), (1.3, 3, None), (1.0, 8, 0.5)]:
    case_logits = gen.normal(size=12) * 2
    hf_scores = TemperatureLogitsWarper(temperature)(None, torch.tensor(case_logits)[None])
    if top_k is not None:
        hf_scores = TopKLogitsWarper(top_k)(None, hf_scores)
    if top_p is not None:
        hf_scores = TopPLogitsWarper(top_p)(None, hf_scores)
    hf_cases.append((case_logits, dict(temperature=temperature, top_k=top_k, top_p=top_p), torch.softmax(hf_scores, dim=-1)[0].numpy()))
toy_logits = np.log(np.array([0.5, 0.3, 0.15, 0.05]))
print(f"built {len(hf_cases)} reference cases with Hugging Face logits warpers")

built 4 reference cases with Hugging Face logits warpers


In [40]:
def sampling_probs(logits, temperature=1.0, top_k=None, top_p=None):
    return None  # TODO: temperature → top-k → top-p, return a renormalized probability vector


def sample_token(logits, rng, **kwargs):
    return None  # TODO: draw one token id from sampling_probs(logits, **kwargs)


check("top_k=2", sampling_probs(toy_logits, top_k=2), [0.625, 0.375, 0.0, 0.0])
check("top_p=0.75 keeps 2 tokens", sampling_probs(toy_logits, top_p=0.75), [0.625, 0.375, 0.0, 0.0],
      hint="Keep a token if the cumulative probability BEFORE it is < p.")
check("top_p=0.85 keeps 3 tokens", sampling_probs(toy_logits, top_p=0.85), [0.5 / 0.95, 0.3 / 0.95, 0.15 / 0.95, 0.0])
check("temperature=0.5 sharpens (p² renormalized)", sampling_probs(toy_logits, temperature=0.5), np.array([0.25, 0.09, 0.0225, 0.0025]) / 0.365)
for case_logits, kwargs, reference in hf_cases:
    check(f"vs Hugging Face warpers {kwargs}", sampling_probs(case_logits, **kwargs), reference)
sample_rng = np.random.default_rng(12)
draws = [sample_token(toy_logits, sample_rng, top_p=0.85) for _ in range(20_000)]
check("empirical frequencies match (±0.015)",
      None if draws[0] is None else bool(np.abs(np.bincount(draws, minlength=4) / len(draws) - np.array([0.5, 0.3, 0.15, 0.0]) / 0.95).max() < 0.015), True,
      hint="rng.choice(len(probs), p=probs)")

⏳ top_k=2: not attempted yet — replace None with your answer.
⏳ top_p=0.75 keeps 2 tokens: not attempted yet — replace None with your answer.
⏳ top_p=0.85 keeps 3 tokens: not attempted yet — replace None with your answer.
⏳ temperature=0.5 sharpens (p² renormalized): not attempted yet — replace None with your answer.
⏳ vs Hugging Face warpers {'temperature': 1.0, 'top_k': 5, 'top_p': 0.8}: not attempted yet — replace None with your answer.
⏳ vs Hugging Face warpers {'temperature': 0.7, 'top_k': None, 'top_p': 0.9}: not attempted yet — replace None with your answer.
⏳ vs Hugging Face warpers {'temperature': 1.3, 'top_k': 3, 'top_p': None}: not attempted yet — replace None with your answer.
⏳ vs Hugging Face warpers {'temperature': 1.0, 'top_k': 8, 'top_p': 0.5}: not attempted yet — replace None with your answer.
⏳ empirical frequencies match (±0.015): not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def sampling_probs(logits, temperature=1.0, top_k=None, top_p=None):
    z = np.asarray(logits, dtype=float) / temperature
    if top_k is not None:
        kth_largest = np.sort(z)[-top_k]
        z = np.where(z < kth_largest, -np.inf, z)                  # ties with the k-th value survive, like HF
    probs = np.exp(z - z.max())
    probs /= probs.sum()
    if top_p is not None:
        order = np.argsort(-probs)
        mass_before = np.cumsum(probs[order]) - probs[order]       # probability of tokens ranked above each token
        remove = np.zeros(len(probs), dtype=bool)
        remove[order] = mass_before >= top_p                       # the most likely token is always kept
        probs = np.where(remove, 0.0, probs)
        probs /= probs.sum()
    return probs


def sample_token(logits, rng, **kwargs):
    probs = sampling_probs(logits, **kwargs)
    return int(rng.choice(len(probs), p=probs))


check("top_k=2", sampling_probs(toy_logits, top_k=2), [0.625, 0.375, 0.0, 0.0])
check("top_p=0.75 keeps 2 tokens", sampling_probs(toy_logits, top_p=0.75), [0.625, 0.375, 0.0, 0.0])
check("top_p=0.85 keeps 3 tokens", sampling_probs(toy_logits, top_p=0.85), [0.5 / 0.95, 0.3 / 0.95, 0.15 / 0.95, 0.0])
check("temperature=0.5 sharpens (p² renormalized)", sampling_probs(toy_logits, temperature=0.5), np.array([0.25, 0.09, 0.0225, 0.0025]) / 0.365)
for case_logits, kwargs, reference in hf_cases:
    check(f"vs Hugging Face warpers {kwargs}", sampling_probs(case_logits, **kwargs), reference)
sample_rng = np.random.default_rng(12)
draws = [sample_token(toy_logits, sample_rng, top_p=0.85) for _ in range(20_000)]
check("empirical frequencies match (±0.015)",
      bool(np.abs(np.bincount(draws, minlength=4) / len(draws) - np.array([0.5, 0.3, 0.15, 0.0]) / 0.95).max() < 0.015), True)
```

Avoid examples where the cumulative sum lands *exactly* on p (e.g. 0.5 + 0.3 vs p = 0.8): floating-point rounding decides the outcome, which is why the checks use 0.75 and 0.85.

**Complexity:** O(V log V) per step for vocabulary size V because of the sort (top-k alone can use `argpartition` in O(V)); O(V) memory.
</details>

#### 🎤 Follow-up questions

**Q36. How do temperature, top-k, and top-p differ, and what does temperature 0 mean?**

<details><summary>Show answer</summary>

- **30-second answer:** Temperature rescales logits: T < 1 sharpens the distribution, T > 1 flattens it, and T → 0 becomes greedy decoding (always the argmax — APIs treat 0 as greedy rather than dividing by zero). Top-k keeps a fixed number of candidates; top-p keeps the smallest set whose probability reaches p, so the candidate count adapts — few when the model is confident, many when it's unsure.
- **Go deeper:** Typical settings: temperature 0 for extraction/classification; ~0.7 with top-p 0.9–0.95 for creative text. Repetition/frequency penalties modify logits before sampling. Even greedy decoding isn't always bit-reproducible on GPUs because batching changes floating-point kernels.
- **❌ Common wrong answer:** "top-p = 0.9 keeps the top 90% of the vocabulary."

</details>

**Q37. When would you use beam search instead of sampling?**

<details><summary>Show answer</summary>

- **30-second answer:** Beam search keeps the B highest-likelihood partial sequences, which suits short outputs with one "correct" answer — translation, speech recognition, structured generation. For open-ended text, maximizing likelihood produces bland, repetitive output (Holtzman et al., 2019), so chat models sample with temperature/top-p. Beam search also costs about B× the compute.
- **Go deeper:** Length normalization is needed because longer sequences accumulate lower log-probabilities. Constrained decoding (grammars, JSON schemas) restricts which tokens are allowed at each step and works with either greedy or sampling.
- **❌ Common wrong answer:** "Beam search always gives better text because it finds more probable sequences."

</details>

## 4. AI-Engineering Utilities 🟡

AI-engineer and LLM-application loops increasingly replace a LeetCode round with "build a small piece of a real AI system": tokenization, retrieval scoring, fusing rankings, chunking documents, caching, and rate limiting API calls. The grading is the same — correctness, edge cases, complexity — but the vocabulary is RAG and LLM serving.

> 💡 Review: [Transformers From Scratch](../04_Deep_Learning/03_Transformers_From_Scratch.ipynb) (BPE) · [Embeddings and Semantic Search](../05_NLP/02_Embeddings_and_Semantic_Search.ipynb) (BM25, RRF, retrieval metrics) · [RAG From Scratch](../08_Generative_AI_LLM/02_RAG_From_Scratch.ipynb) (chunking) · [Python Internals and Concurrency](../00_Foundations/05_Python_Internals_and_Concurrency.ipynb) (data structures, locks)

### Drill 4.1 — One byte-pair-encoding (BPE) merge step 🟡 (20 min)

**Problem.** A corpus is given as `{tuple_of_symbols: word_count}`. Implement `bpe_merge_step(word_freqs)` that
1. counts every adjacent symbol pair, weighted by the word's count,
2. picks the most frequent pair — ties broken by the lexicographically **smallest** pair,
3. returns `(best_pair, new_word_freqs)` with that pair merged everywhere (left to right, non-overlapping), or `(None, word_freqs)` if no pairs exist.

**Example:** the classic corpus from Sennrich et al. (2016) below; `</w>` marks the end of a word.

In [41]:
def bpe_merge_step(word_freqs):
    return None  # TODO: return (best_pair, new_word_freqs)


corpus = {("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2,
          ("n", "e", "w", "e", "s", "t", "</w>"): 6, ("w", "i", "d", "e", "s", "t", "</w>"): 3}
step = bpe_merge_step(corpus)
check("first merge (count 9, tie → smallest pair)", None if step is None else step[0], ("e", "s"),
      hint="('e','s'), ('s','t') and ('t','</w>') all occur 9 times; min(pairs, key=lambda p: (-count[p], p)).")
check("vocabulary after one merge", None if step is None else step[1],
      {("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2,
       ("n", "e", "w", "es", "t", "</w>"): 6, ("w", "i", "d", "es", "t", "</w>"): 3})
merges, vocab = [], corpus
for _ in range(4):
    step = bpe_merge_step(vocab)
    if step is None:
        merges = None
        break
    pair, vocab = step
    merges.append(pair)
check("first four merges", merges, [("e", "s"), ("es", "t"), ("est", "</w>"), ("l", "o")])
overlap = bpe_merge_step({("a", "a", "a"): 1})
check("'a a a' merges left to right into 'aa a'", None if overlap is None else overlap[1], {("aa", "a"): 1},
      hint="After merging at position i, jump to i + 2.")

⏳ first merge (count 9, tie → smallest pair): not attempted yet — replace None with your answer.
⏳ vocabulary after one merge: not attempted yet — replace None with your answer.
⏳ first four merges: not attempted yet — replace None with your answer.
⏳ 'a a a' merges left to right into 'aa a': not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
from collections import Counter


def bpe_merge_step(word_freqs):
    pair_counts = Counter()
    for word, freq in word_freqs.items():
        for pair in zip(word, word[1:]):                               # adjacent pairs, weighted by word count
            pair_counts[pair] += freq
    if not pair_counts:
        return None, word_freqs
    best = min(pair_counts, key=lambda pair: (-pair_counts[pair], pair))   # highest count, then smallest pair
    merged = {}
    for word, freq in word_freqs.items():
        out, i = [], 0
        while i < len(word):
            if i + 1 < len(word) and (word[i], word[i + 1]) == best:
                out.append(word[i] + word[i + 1])                      # merge and skip the partner
                i += 2
            else:
                out.append(word[i])
                i += 1
        merged[tuple(out)] = merged.get(tuple(out), 0) + freq
    return best, merged


corpus = {("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2,
          ("n", "e", "w", "e", "s", "t", "</w>"): 6, ("w", "i", "d", "e", "s", "t", "</w>"): 3}
step = bpe_merge_step(corpus)
check("first merge (count 9, tie → smallest pair)", step[0], ("e", "s"))
check("vocabulary after one merge", step[1],
      {("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2,
       ("n", "e", "w", "es", "t", "</w>"): 6, ("w", "i", "d", "es", "t", "</w>"): 3})
merges, vocab = [], corpus
for _ in range(4):
    pair, vocab = bpe_merge_step(vocab)
    merges.append(pair)
check("first four merges", merges, [("e", "s"), ("es", "t"), ("est", "</w>"), ("l", "o")])
check("'a a a' merges left to right into 'aa a'", bpe_merge_step({("a", "a", "a"): 1})[1], {("aa", "a"): 1})
```

**Complexity:** one step is O(S) to count pairs and O(S) to rewrite the corpus, where S is the total number of symbols in the distinct words (plus O(P) to pick the best of P pairs). Learning V merges naively costs O(V·S); production trainers update only the pair counts touched by each merge.
</details>

#### 🎤 Follow-up questions

**Q38. Why do LLMs use subword tokenization rather than words or characters?**

<details><summary>Show answer</summary>

- **30-second answer:** Word vocabularies explode and can't represent unseen words; character sequences are very long (attention is O(T²)) and each token carries little meaning. Subword methods (BPE, WordPiece, Unigram) keep frequent words as single tokens and split rare words into pieces, with a fixed vocabulary of roughly 30k–200k tokens.
- **Go deeper:** Byte-level BPE (GPT-2 onward) starts from the 256 byte values, so any string can be encoded and there is no unknown token. Tokenization drives cost and context usage (APIs bill per token), explains quirks with spelling and arithmetic, and is uneven across languages — some scripts need many more tokens per sentence.
- **❌ Common wrong answer:** "A token is a word."

</details>

**Q39. After BPE is trained, how is new text tokenized?**

<details><summary>Show answer</summary>

- **30-second answer:** Pre-tokenize the text (e.g. split on spaces and punctuation with a regex), turn each piece into base symbols (bytes), then repeatedly apply the learned merges **in the order they were learned** — at each step merge the adjacent pair with the lowest merge rank — until no learned merge applies; finally map symbols to ids.
- **Go deeper:** Pre-tokenization stops merges from crossing word boundaries. Encoding is deterministic, which is why the same text always costs the same number of tokens. Hugging Face `tokenizers` and OpenAI's `tiktoken` implement this in Rust for speed.
- **❌ Common wrong answer:** "Greedily match the longest token in the vocabulary" — that's how WordPiece tokenizes at inference, not BPE.

</details>

### Drill 4.2 — BM25 scoring 🟡 (20 min)

**Problem.** Implement `bm25_scores(query_tokens, docs_tokens, k1=1.5, b=0.75)` returning one score per document, summing over the **unique** query terms:

$$\text{score}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{tf_{t,d}\,(k_1 + 1)}{tf_{t,d} + k_1\left(1 - b + b\,\frac{\lvert d\rvert}{\text{avgdl}}\right)}, \qquad \text{idf}(t) = \ln\!\left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)$$

**Checker:** the `bm25s` library's Lucene variant uses the same IDF but drops the constant $(k_1 + 1)$ factor (it doesn't change the ranking), so we compare with its scores × $(k_1+1)$.

In [42]:
import bm25s

docs_text = ["the cat sat on the mat",
             "dogs and cats living together",
             "the quick brown fox jumps over the lazy dog",
             "a cat and a dog are friends",
             "machine learning with cats"]
docs_tokens = [doc.split() for doc in docs_text]
query_tokens = ["cat", "dog"]
k1, b = 1.5, 0.75
bm25_index = bm25s.BM25(k1=k1, b=b, method="lucene")
bm25_index.index(docs_tokens, show_progress=False)
bm25s_scores = bm25_index.get_scores(query_tokens)
print("bm25s (Lucene) scores:", bm25s_scores.round(4))

bm25s (Lucene) scores: [0.3553 0.     0.291  0.6619 0.    ]


In [43]:
def bm25_scores(query_tokens, docs_tokens, k1=1.5, b=0.75):
    return None  # TODO: one BM25 score per document


bm25_ours = bm25_scores(query_tokens, docs_tokens, k1, b)
check("BM25 vs bm25s × (k1 + 1)", bm25_ours, bm25s_scores * (k1 + 1),
      hint="Count df per term across documents, tf per document, and the average document length.")
check("same ranking as bm25s", None if bm25_ours is None else np.argsort(-bm25_ours, kind="stable"),
      np.argsort(-bm25s_scores, kind="stable"))

⏳ BM25 vs bm25s × (k1 + 1): not attempted yet — replace None with your answer.
⏳ same ranking as bm25s: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
import math
from collections import Counter


def bm25_scores(query_tokens, docs_tokens, k1=1.5, b=0.75):
    n_docs = len(docs_tokens)
    avgdl = sum(len(doc) for doc in docs_tokens) / n_docs
    term_counts = [Counter(doc) for doc in docs_tokens]                # tf per document
    scores = np.zeros(n_docs)
    for term in set(query_tokens):
        df = sum(term in counts for counts in term_counts)
        if df == 0:
            continue
        idf = math.log(1 + (n_docs - df + 0.5) / (df + 0.5))            # Lucene's IDF: always positive
        for i, counts in enumerate(term_counts):
            tf = counts[term]                                           # Counter returns 0 for missing terms
            length_norm = 1 - b + b * len(docs_tokens[i]) / avgdl
            scores[i] += idf * tf * (k1 + 1) / (tf + k1 * length_norm)
    return scores


bm25_ours = bm25_scores(query_tokens, docs_tokens, k1, b)
check("BM25 vs bm25s × (k1 + 1)", bm25_ours, bm25s_scores * (k1 + 1))
check("same ranking as bm25s", np.argsort(-bm25_ours, kind="stable"), np.argsort(-bm25s_scores, kind="stable"))
for text, score in sorted(zip(docs_text, bm25_ours), key=lambda pair: -pair[1]):
    print(f"{score:6.3f}  {text}")
```

Notice that "dogs and cats living together" scores 0: without stemming, `cats` ≠ `cat`. Real systems normalize tokens (lower-casing, stemming or lemmatization) before indexing.

**Complexity:** this version is O(|q| · N) scoring after O(total tokens) counting. An **inverted index** (term → list of (doc, tf)) makes a query cost O(total postings of its terms), which is how search engines scale.
</details>

#### 🎤 Follow-up questions

**Q40. What do BM25's k1 and b control?**

<details><summary>Show answer</summary>

- **30-second answer:** k1 controls term-frequency saturation: repeating a word adds less and less score (k1 = 0 ignores tf entirely; a larger k1 makes tf count more before it saturates). b controls document-length normalization: b = 0 ignores length, b = 1 fully normalizes, so long documents don't win just by containing more words. Common defaults are k1 ≈ 1.2–2.0 and b = 0.75.
- **Go deeper:** IDF rewards rare terms; Lucene's variant, ln(1 + (N − df + 0.5)/(df + 0.5)), is always positive, whereas the classic Robertson–Spärck Jones IDF goes negative for terms in more than half the documents. Tune k1 and b on a labelled validation set if retrieval quality matters.
- **❌ Common wrong answer:** "k1 is the number of results to return."

</details>

**Q41. When does BM25 beat dense (embedding) retrieval, and why do production RAG systems use hybrid search?**

<details><summary>Show answer</summary>

- **30-second answer:** BM25 wins on exact matches — product codes, error messages, names, rare jargon, and domains the embedding model never saw — and needs no training or GPU. Dense retrieval wins on paraphrases and synonyms. Because they fail on different queries, combining them (e.g. with reciprocal rank fusion) and then reranking the top candidates is a strong default.
- **Go deeper:** Measure with recall@k / nDCG@k on a labelled query set rather than intuition. Hybrid search is built into Elasticsearch/OpenSearch and most vector databases. Learned sparse models (e.g. SPLADE) sit in between: sparse, index-friendly, but trained.
- **❌ Common wrong answer:** "Embeddings made keyword search obsolete."

</details>

### Drill 4.3 — Reciprocal rank fusion (RRF) 🟢 (10 min)

**Problem.** Implement `reciprocal_rank_fusion(rankings, k=60)` that fuses several ranked lists of document ids:

$$\text{RRF}(d) = \sum_{\text{lists } L \ni d} \frac{1}{k + \text{rank}_L(d)}, \qquad \text{ranks start at } 1$$

Return a list of `(doc_id, score)` sorted by score (highest first), ties broken by `doc_id`. A document missing from a list gets nothing from that list.

In [44]:
def reciprocal_rank_fusion(rankings, k=60):
    return None  # TODO: list of (doc_id, score), best first


bm25_ranking = ["a", "b", "c"]
dense_ranking = ["c", "a", "d"]
fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking])
check("fused order", None if fused is None else [doc for doc, _ in fused], ["a", "c", "b", "d"],
      hint="Accumulate 1 / (k + rank) per document in a dict, then sort by (-score, doc).")
check("fused scores", None if fused is None else [score for _, score in fused], [1 / 61 + 1 / 62, 1 / 63 + 1 / 61, 1 / 62, 1 / 63])

⏳ fused order: not attempted yet — replace None with your answer.
⏳ fused scores: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))


bm25_ranking = ["a", "b", "c"]
dense_ranking = ["c", "a", "d"]
fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking])
check("fused order", [doc for doc, _ in fused], ["a", "c", "b", "d"])
check("fused scores", [score for _, score in fused], [1 / 61 + 1 / 62, 1 / 63 + 1 / 61, 1 / 62, 1 / 63])
```

`a` (ranks 1 and 2) beats `c` (ranks 3 and 1): being near the top of **both** lists matters more than being first in one.

**Complexity:** O(L·n) to accumulate scores over L lists of length n, plus O(m log m) to sort the m unique documents.
</details>

#### 🎤 Follow-up question

**Q42. Why fuse ranks instead of adding the raw scores, and what does the constant k do?**

<details><summary>Show answer</summary>

- **30-second answer:** BM25 scores are unbounded and vary per query, while cosine similarities sit in [−1, 1]; adding them requires normalization and a tuned weight that breaks when either score distribution shifts. RRF uses only ranks, so it's scale-free and robust. k = 60 (the value from Cormack et al., 2009) flattens the curve — rank 1 earns 1/61 and rank 10 earns 1/70 — so agreement across lists outweighs a single first place.
- **Go deeper:** Weighted RRF multiplies each list's contribution by a weight tuned on validation queries. If you have relevance labels and latency budget, a cross-encoder reranker over the fused top-k usually beats any unsupervised fusion.
- **❌ Common wrong answer:** "RRF needs calibrated probabilities from each retriever."

</details>

### Drill 4.4 — Text chunking with overlap 🟢 (10 min)

**Problem.** Implement `chunk_tokens(tokens, chunk_size, overlap)` returning a list of chunks (lists), where consecutive chunks share `overlap` tokens.

**Constraints:** raise `ValueError` unless `0 <= overlap < chunk_size` (otherwise the loop never advances); an empty input gives `[]`; input shorter than a chunk gives one chunk; stop as soon as a chunk reaches the end — no tiny trailing chunk that's already contained in the previous one.

In [45]:
words = "retrieval augmented generation splits long documents into small overlapping chunks".split()
print(len(words), "words")


def as_text(chunks):
    """Join each chunk back into a string so results are easy to read and compare."""
    return None if chunks is None else [" ".join(chunk) for chunk in chunks]

10 words


In [46]:
def chunk_tokens(tokens, chunk_size, overlap):
    return None  # TODO: list of chunks; ValueError unless 0 <= overlap < chunk_size


check("size 4, overlap 1", as_text(chunk_tokens(words, 4, 1)),
      ["retrieval augmented generation splits", "splits long documents into", "into small overlapping chunks"],
      hint="step = chunk_size − overlap; for start in range(0, len(tokens), step) … break when start + chunk_size >= len(tokens).")
check("size 4, overlap 2", as_text(chunk_tokens(words, 4, 2)),
      ["retrieval augmented generation splits", "generation splits long documents",
       "long documents into small", "into small overlapping chunks"])
check("short input → one chunk", as_text(chunk_tokens(words[:3], 4, 1)), ["retrieval augmented generation"])
check("empty input → no chunks", as_text(chunk_tokens([], 4, 1)), [])
if chunk_tokens(words, 4, 1) is None:
    check("overlap >= chunk_size raises ValueError", None, True)
else:
    try:
        chunk_tokens(words, 4, 4)
        raised = False
    except ValueError:
        raised = True
    check("overlap >= chunk_size raises ValueError", raised, True, hint="Validate the arguments before looping.")

⏳ size 4, overlap 1: not attempted yet — replace None with your answer.
⏳ size 4, overlap 2: not attempted yet — replace None with your answer.
⏳ short input → one chunk: not attempted yet — replace None with your answer.
⏳ empty input → no chunks: not attempted yet — replace None with your answer.
⏳ overlap >= chunk_size raises ValueError: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def chunk_tokens(tokens, chunk_size, overlap):
    if not 0 <= overlap < chunk_size:
        raise ValueError("need 0 <= overlap < chunk_size")
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(tokens), step):
        chunks.append(tokens[start:start + chunk_size])
        if start + chunk_size >= len(tokens):          # this chunk reached the end → stop
            break
    return chunks


check("size 4, overlap 1", as_text(chunk_tokens(words, 4, 1)),
      ["retrieval augmented generation splits", "splits long documents into", "into small overlapping chunks"])
check("size 4, overlap 2", as_text(chunk_tokens(words, 4, 2)),
      ["retrieval augmented generation splits", "generation splits long documents",
       "long documents into small", "into small overlapping chunks"])
check("short input → one chunk", as_text(chunk_tokens(words[:3], 4, 1)), ["retrieval augmented generation"])
check("empty input → no chunks", as_text(chunk_tokens([], 4, 1)), [])
try:
    chunk_tokens(words, 4, 4)
    raised = False
except ValueError:
    raised = True
check("overlap >= chunk_size raises ValueError", raised, True)
```

**Complexity:** O(n · chunk_size / step) time and memory — the total size of the output. Store `(start, end)` offsets instead of copies when you need to cite the source span.
</details>

#### 🎤 Follow-up question

**Q43. How do you choose chunk size and overlap for a RAG system?**

<details><summary>Show answer</summary>

- **30-second answer:** It's a trade-off: small chunks give precise matches but lose surrounding context; large chunks keep context but dilute the embedding and spend more prompt tokens. Overlap (often ~10–20%) keeps answers from being cut at a boundary, at the cost of more index entries and duplicate results. Pick sizes empirically with a retrieval evaluation (recall@k on real questions), within the embedding model's maximum input length.
- **Go deeper:** Measure chunks in **tokens** (models limit tokens, not characters). Respect structure — split on headings, paragraphs, and code blocks rather than mid-sentence — and attach metadata (title, section, URL) to each chunk. "Small-to-big" retrieval indexes small chunks but returns their parent section.
- **❌ Common wrong answer:** "Use the biggest chunks possible — long-context models can read everything."

</details>

### Drill 4.5 — LRU cache with O(1) operations 🟡 (25 min)

**Problem.** Implement `LRUCache(capacity)` with `get(key)` (return the value and mark the key as recently used, or −1 if missing) and `put(key, value)` (insert or update and mark as recently used; when over capacity, evict the **least recently used** key). Both must be **O(1)**. This is LeetCode 146; many interviewers disallow `OrderedDict`, so aim for a hash map + doubly linked list.

In [47]:
def replay(cache, ops):
    """Run ('put', k, v) / ('get', k) operations and collect the results of the gets."""
    results = []
    for op in ops:
        if op[0] == "put":
            cache.put(op[1], op[2])
        else:
            results.append(cache.get(op[1]))
    return None if None in results else results


leetcode_ops = [("put", 1, 1), ("put", 2, 2), ("get", 1), ("put", 3, 3), ("get", 2),
                ("put", 4, 4), ("get", 1), ("get", 3), ("get", 4)]
update_ops = [("put", 1, 1), ("put", 2, 2), ("put", 1, 10), ("put", 3, 3), ("get", 2), ("get", 1), ("get", 3)]
print(f"{len(leetcode_ops)} + {len(update_ops)} test operations ready")

9 + 7 test operations ready


In [48]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        # TODO: choose data structures so that get and put are O(1)

    def get(self, key):
        return None  # TODO: value if present (and mark as recently used), else -1

    def put(self, key, value):
        pass  # TODO: insert/update, mark as recently used, evict the LRU key when over capacity


check("LeetCode 146 example", replay(LRUCache(2), leetcode_ops), [1, -1, -1, 3, 4],
      hint="get(1) makes 1 recent, so put(3) evicts 2.")
check("updating a key also makes it recent", replay(LRUCache(2), update_ops), [-1, 10, 3])

⏳ LeetCode 146 example: not attempted yet — replace None with your answer.
⏳ updating a key also makes it recent: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class _Node:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key=None, value=None):
        self.key, self.value, self.prev, self.next = key, value, None, None


class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.nodes = {}                                   # key → node, for O(1) lookup
        self.head, self.tail = _Node(), _Node()          # sentinels: head.next = most recent, tail.prev = least recent
        self.head.next, self.tail.prev = self.tail, self.head

    def _unlink(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _push_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        node = self.nodes.get(key)
        if node is None:
            return -1
        self._unlink(node)
        self._push_front(node)
        return node.value

    def put(self, key, value):
        if key in self.nodes:
            node = self.nodes[key]
            node.value = value
            self._unlink(node)
        else:
            node = _Node(key, value)
            self.nodes[key] = node
            if len(self.nodes) > self.capacity:
                lru = self.tail.prev                      # least recently used real node
                self._unlink(lru)
                del self.nodes[lru.key]
        self._push_front(node)


check("LeetCode 146 example", replay(LRUCache(2), leetcode_ops), [1, -1, -1, 3, 4])
check("updating a key also makes it recent", replay(LRUCache(2), update_ops), [-1, 10, 3])
```

The one-liner alternative: `collections.OrderedDict` with `move_to_end(key)` on access and `popitem(last=False)` to evict.

**Complexity:** O(1) time for `get` and `put`; O(capacity) memory.
</details>

#### 🎤 Follow-up questions

**Q44. Why do you need both a hash map and a doubly linked list?**

<details><summary>Show answer</summary>

- **30-second answer:** The hash map finds a key's node in O(1); the doubly linked list keeps recency order and lets you move or remove a node in O(1) because each node knows both neighbours. A list alone needs O(n) to find a key; a hash map alone can't tell you which key is least recently used without an O(n) scan.
- **Go deeper:** Sentinel head/tail nodes remove all the "is this the first/last node?" special cases. In a multi-threaded server, guard both structures with one lock, since `get` also mutates the order. LFU caches need frequency buckets instead of one list.
- **❌ Common wrong answer:** "Store keys in a Python list and call `list.remove(key)` on access" — that's O(n).

</details>

**Q45. Where does caching pay off in ML and LLM systems, and what can go wrong?**

<details><summary>Show answer</summary>

- **30-second answer:** Cache expensive, repeatable work: embeddings of frequently seen texts, feature lookups, model predictions for identical inputs, and LLM responses for identical requests; inside LLM serving, KV/prefix caches reuse computation for shared prompt prefixes. The cache key must include everything that affects the output — model version, prompt, and generation parameters — with TTLs and invalidation on model updates.
- **Go deeper:** `functools.lru_cache` is per-process and keeps references to arguments (on methods it keeps `self` alive), so use Redis or similar across replicas. "Semantic" caches that reuse answers for *similar* prompts raise hit rates but can return wrong answers — tune and monitor the similarity threshold. Mind privacy: never share cached responses across users when prompts contain personal data.
- **❌ Common wrong answer:** "Key the cache on the prompt text alone."

</details>

### Drill 4.6 — Token-bucket rate limiter 🟡 (20 min)

**Problem.** Implement `TokenBucket(capacity, refill_rate, clock)` with `allow(cost=1.0) -> bool`. The bucket starts full, refills continuously at `refill_rate` tokens per second up to `capacity`, and a request is allowed only if enough tokens remain (then they are spent).

**Constraints:** O(1) per call — refill lazily from the elapsed time; take the clock as a parameter so tests are deterministic (never `time.sleep` in a test).

In [49]:
class FakeClock:
    """A controllable clock: set .now, and the bucket reads it by calling the clock."""

    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

In [50]:
class TokenBucket:
    def __init__(self, capacity, refill_rate, clock):
        self.capacity, self.refill_rate, self.clock = capacity, refill_rate, clock
        # TODO: start full and remember when you last refilled

    def allow(self, cost=1.0):
        return None  # TODO: refill from elapsed time (capped at capacity), then spend `cost` tokens if available


clock = FakeClock()
bucket = TokenBucket(capacity=3, refill_rate=1.0, clock=clock)
decisions = []
for t in [0, 0, 0, 0, 0.5, 1.0, 1.0, 10, 10, 10, 10]:
    clock.now = t
    decisions.append(bucket.allow())
check("burst of 3, then 1 token/second, capped at capacity", None if None in decisions else decisions,
      [True, True, True, False, False, True, False, True, True, True, False],
      hint="tokens = min(capacity, tokens + (now − last) · rate); last = now; allow if tokens >= cost.")
check("a cost above capacity is never allowed", TokenBucket(3, 1.0, FakeClock()).allow(cost=5), False)

⏳ burst of 3, then 1 token/second, capped at capacity: not attempted yet — replace None with your answer.
⏳ a cost above capacity is never allowed: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class TokenBucket:
    def __init__(self, capacity, refill_rate, clock):
        self.capacity, self.refill_rate, self.clock = capacity, refill_rate, clock
        self.tokens = float(capacity)                     # start full → allows an initial burst
        self.last = clock()

    def allow(self, cost=1.0):
        now = self.clock()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.refill_rate)   # lazy refill
        self.last = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False


clock = FakeClock()
bucket = TokenBucket(capacity=3, refill_rate=1.0, clock=clock)
decisions = []
for t in [0, 0, 0, 0, 0.5, 1.0, 1.0, 10, 10, 10, 10]:
    clock.now = t
    decisions.append(bucket.allow())
check("burst of 3, then 1 token/second, capped at capacity", decisions,
      [True, True, True, False, False, True, False, True, True, True, False])
check("a cost above capacity is never allowed", TokenBucket(3, 1.0, FakeClock()).allow(cost=5), False)
```

At t = 0.5 only half a token has refilled, so the request is rejected; at t = 10 the bucket is capped at 3 tokens, not 9. In production use `time.monotonic()` as the clock (wall-clock time can jump backwards).

**Complexity:** O(1) time per call and O(1) memory per bucket (one bucket per user or API key).
</details>

#### 🎤 Follow-up questions

**Q46. Token bucket vs leaky bucket vs fixed-window vs sliding-window rate limiting?**

<details><summary>Show answer</summary>

- **30-second answer:** A token bucket enforces an average rate but allows bursts up to its capacity. A leaky bucket (used as a queue) smooths traffic to a constant output rate. A fixed-window counter ("100 per minute") is simplest but allows up to 2× the limit around window boundaries. Sliding-window logs or counters fix that boundary burst at the cost of more memory or approximation.
- **Go deeper:** Pick by the behaviour you want: user-facing APIs usually tolerate short bursts (token bucket); protecting a fragile downstream service may call for smoothing (leaky bucket). LLM providers typically enforce both requests-per-minute and tokens-per-minute, so the "cost" of a call is its token count.
- **❌ Common wrong answer:** "They're all the same thing with different names."

</details>

**Q47. How do you rate-limit across many servers, and how should a client handle rate-limit errors from an LLM API?**

<details><summary>Show answer</summary>

- **30-second answer:** In-memory buckets only limit one process. For a fleet, keep bucket state in a shared store such as Redis and update it atomically (a Lua script or transaction) using the store's clock, or enforce limits at an API gateway. Clients should treat HTTP 429 as retryable: exponential backoff with jitter, honour a `Retry-After` header, and cap retries.
- **Go deeper:** Within one process, protect the bucket with a lock (threads) or keep it on one event loop (asyncio). Pre-count tokens before sending LLM requests to stay under tokens-per-minute, and use a queue with bounded concurrency rather than firing everything at once.
- **❌ Common wrong answer:** "Add `time.sleep(1)` between calls."

</details>

## 🏋️ Timed Drills

A **mock round** is 45 minutes, 2–3 problems, no peeking. It trains the thing the drills alone can't: pacing and talking while you code.

**How to run one:**
1. Open a blank notebook (or a plain editor with no autocomplete if your interviews use CoderPad) and start a 45-minute timer.
2. Talk out loud the whole time — record yourself on your phone. Listening back is uncomfortable and extremely effective.
3. When time is up, paste each solution into the matching drill cell above and run the checker.
4. Score yourself with the rubric, write down the lowest dimension, and redo those drills untimed.

### 📏 Self-review rubric (score each 0–2, total /14)

| Dimension | 0 — not yet | 1 — partly | 2 — interview-ready |
|---|---|---|---|
| **Clarify & plan** | Started typing immediately | Asked a few questions | Stated inputs, outputs, constraints, edge cases, and the plan before coding |
| **Correctness** | Doesn't run or wrong output | Works on the happy path | Passes every check, including edge cases |
| **Testing** | Never ran it | Ran one example | Hand-checked example + edge case + library/brute-force comparison |
| **Vectorization & data structures** | Python loops over the data | Partly vectorized | Vectorized where it matters; right data structure (heap, dict, linked list) |
| **Numerical stability** | `nan`/`inf` possible | Aware but not handled | Stable by construction (shift, log space, clip, eps) |
| **Complexity** | Not mentioned | Time only, or vague | Time and memory, the bottleneck, and how to scale |
| **Communication** | Silent or rambling | Explained after the fact | Narrated decisions and trade-offs while coding |

**12–14:** ready for the real thing · **8–11:** close — redo the drills behind your weakest dimension · **≤ 7:** go back through the sections untimed first.

### ⏱️ Mock Round A — Applied ML / Data Science (45 min)

| Minutes | Task |
|---|---|
| 0–5 | Read all three tasks; state inputs, outputs, and edge cases for each out loud |
| 5–18 | Redo **Drill 2.8** (precision/recall/F1 + rank-based ROC-AUC) from a blank cell |
| 18–30 | Redo **Drill 2.2** (stable sigmoid + logistic regression with L2) |
| 30–45 | **New problem A** below: multiclass confusion matrix and macro-F1 |

#### New problem A — Confusion matrix and macro-F1 🟡 (15 min)

**Problem.** Implement `confusion_matrix_np(y_true, y_pred, n_classes)` where `cm[i, j]` counts samples of true class i predicted as j, and `macro_f1(y_true, y_pred, n_classes)` — the unweighted mean of per-class F1 (a class with no support and no predictions gets F1 = 0).

**Test data:** real predictions — a Gaussian Naive Bayes model on the scikit-learn digits dataset (1,797 8×8 images, 10 classes).

In [51]:
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix

digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
yd_pred = GaussianNB().fit(Xd_tr, yd_tr).predict(Xd_te)
print(f"GaussianNB accuracy on digits: {(yd_pred == yd_te).mean():.3f}")

GaussianNB accuracy on digits: 0.848


In [52]:
def confusion_matrix_np(y_true, y_pred, n_classes):
    return None  # TODO: (n_classes, n_classes) int matrix, rows = true class, columns = predicted class


def macro_f1(y_true, y_pred, n_classes):
    return None  # TODO: unweighted mean of per-class F1


check("confusion matrix vs sklearn", confusion_matrix_np(yd_te, yd_pred, 10), confusion_matrix(yd_te, yd_pred, labels=list(range(10))),
      hint="np.add.at(cm, (y_true, y_pred), 1) — plain cm[y_true, y_pred] += 1 counts repeated pairs only once.")
check("macro-F1 vs sklearn", macro_f1(yd_te, yd_pred, 10), f1_score(yd_te, yd_pred, average="macro"),
      hint="Per class: F1 = 2·TP / (2·TP + FP + FN), with TP on the diagonal, FP = column sum − TP, FN = row sum − TP.")

⏳ confusion matrix vs sklearn: not attempted yet — replace None with your answer.
⏳ macro-F1 vs sklearn: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def confusion_matrix_np(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)                     # unbuffered: every repeated (i, j) pair counts
    return cm


def macro_f1(y_true, y_pred, n_classes):
    cm = confusion_matrix_np(y_true, y_pred, n_classes)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp                               # predicted as c, truly something else
    fn = cm.sum(axis=1) - tp                               # truly c, predicted as something else
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros(n_classes), where=denominator > 0)
    return float(f1.mean())


check("confusion matrix vs sklearn", confusion_matrix_np(yd_te, yd_pred, 10), confusion_matrix(yd_te, yd_pred, labels=list(range(10))))
check("macro-F1 vs sklearn", macro_f1(yd_te, yd_pred, 10), f1_score(yd_te, yd_pred, average="macro"))
print(f"micro-F1 {f1_score(yd_te, yd_pred, average='micro'):.4f} == accuracy {(yd_te == yd_pred).mean():.4f} | "
      f"macro-F1 {macro_f1(yd_te, yd_pred, 10):.4f} | weighted-F1 {f1_score(yd_te, yd_pred, average='weighted'):.4f}")
```

`np.bincount(n_classes * y_true + y_pred, minlength=n_classes ** 2).reshape(n_classes, n_classes)` is an equally good one-liner.

**Complexity:** O(n + C²) time, O(C²) memory.
</details>

**Q48. Macro-, micro-, or weighted-F1 — which do you report for an imbalanced multiclass problem?**

<details><summary>Show answer</summary>

- **30-second answer:** Macro-F1 averages per-class F1 equally, so a failing minority class drags it down — report it (plus the per-class table) when every class matters. Micro-F1 pools all TP/FP/FN; for single-label multiclass it equals accuracy (the solution prints both), so it's dominated by frequent classes. Weighted-F1 averages by class support, also favouring big classes.
- **Go deeper:** Always show the confusion matrix: it tells you *which* classes are confused, which a single number hides. For multi-label problems micro and macro differ from accuracy and both are commonly reported.
- **❌ Common wrong answer:** "Weighted-F1, because it accounts for imbalance" — it accounts for imbalance by *hiding* the small classes.

</details>

### ⏱️ Mock Round B — Deep Learning / Computer Vision (45 min)

| Minutes | Task |
|---|---|
| 0–5 | Clarify shapes and conventions (box format, mask convention) out loud |
| 5–25 | Redo **Drill 3.6** (scaled dot-product + multi-head attention with a causal mask) |
| 25–40 | **New problem B** below: IoU and non-maximum suppression |
| 40–45 | Answer **Q28** (BatchNorm train vs eval) and **Q34** (why √d_k) out loud, 2 minutes each |

#### New problem B — IoU and non-maximum suppression (NMS) 🟡 (15 min)

**Problem.** Boxes are `(x1, y1, x2, y2)`. Implement `box_iou(box, boxes)` (IoU of one box with each row of `boxes`) and `nms(boxes, scores, iou_threshold)`: repeatedly keep the highest-scoring remaining box and discard every remaining box whose IoU with it is **greater than** the threshold. Return kept indices, highest score first.

$$\text{IoU}(A, B) = \frac{\text{area}(A \cap B)}{\text{area}(A) + \text{area}(B) - \text{area}(A \cap B)}$$

In [53]:
import torchvision

gen = np.random.default_rng(13)
corners = gen.uniform(0, 50, size=(40, 2))
sizes = gen.uniform(5, 30, size=(40, 2))
boxes = np.concatenate([corners, corners + sizes], axis=1)          # synthetic boxes (x1, y1, x2, y2)
box_scores = gen.random(40)
ref_iou = torchvision.ops.box_iou(torch.tensor(boxes[:1]), torch.tensor(boxes)).numpy()[0]
ref_keep = torchvision.ops.nms(torch.tensor(boxes), torch.tensor(box_scores), iou_threshold=0.5).numpy()
print(f"torchvision NMS keeps {len(ref_keep)} of {len(boxes)} boxes")

torchvision NMS keeps 37 of 40 boxes


In [54]:
def box_iou(box, boxes):
    return None  # TODO: IoU between one box and each row of boxes


def nms(boxes, scores, iou_threshold):
    return None  # TODO: kept indices, highest score first


check("IoU hand example", box_iou(np.array([0.0, 0.0, 2.0, 2.0]), np.array([[1.0, 1.0, 3.0, 3.0], [0.0, 0.0, 2.0, 2.0], [5.0, 5.0, 6.0, 6.0]])),
      [1 / 7, 1.0, 0.0], hint="Intersection width = max(0, min(x2) − max(x1)); same for height.")
check("IoU vs torchvision", box_iou(boxes[0], boxes), ref_iou)
check("NMS vs torchvision", nms(boxes, box_scores, 0.5), ref_keep,
      hint="order = argsort(-scores); keep order[0]; drop the rest whose IoU with it > threshold; repeat.")

⏳ IoU hand example: not attempted yet — replace None with your answer.
⏳ IoU vs torchvision: not attempted yet — replace None with your answer.
⏳ NMS vs torchvision: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def box_iou(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    intersection = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)   # 0 when boxes don't overlap
    area_box = (box[2] - box[0]) * (box[3] - box[1])
    area_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return intersection / (area_box + area_boxes - intersection)


def nms(boxes, scores, iou_threshold):
    order = np.argsort(-scores)
    keep = []
    while order.size > 0:
        best = order[0]
        keep.append(int(best))
        rest = order[1:]
        order = rest[box_iou(boxes[best], boxes[rest]) <= iou_threshold]   # suppress heavy overlaps
    return np.array(keep)


check("IoU hand example", box_iou(np.array([0.0, 0.0, 2.0, 2.0]), np.array([[1.0, 1.0, 3.0, 3.0], [0.0, 0.0, 2.0, 2.0], [5.0, 5.0, 6.0, 6.0]])),
      [1 / 7, 1.0, 0.0])
check("IoU vs torchvision", box_iou(boxes[0], boxes), ref_iou)
check("NMS vs torchvision", nms(boxes, box_scores, 0.5), ref_keep)
```

**Complexity:** O(n log n) to sort plus O(n²) IoU computations in the worst case (few suppressions); O(n) memory.
</details>

**Q49. What are the weaknesses of greedy NMS, and what alternatives exist?**

<details><summary>Show answer</summary>

- **30-second answer:** Greedy NMS can delete real objects that overlap heavily (crowds, stacked items), its IoU threshold is a hand-tuned hyperparameter, and the loop is sequential. Alternatives: Soft-NMS lowers overlapping boxes' scores instead of deleting them; class-aware ("batched") NMS runs per class so a person doesn't suppress an overlapping bicycle (`torchvision.ops.batched_nms`); and NMS-free detectors — DETR-style set prediction with Hungarian matching, or YOLOv10's training scheme — remove the step entirely.
- **Go deeper:** Evaluation (mAP) is sensitive to the NMS threshold, so tune it on validation data. GPU implementations vectorize the IoU matrix; the greedy selection is still the bottleneck for thousands of boxes.
- **❌ Common wrong answer:** "NMS runs once across all classes together."

</details>

### ⏱️ Mock Round C — AI / LLM Engineering (45 min)

| Minutes | Task |
|---|---|
| 0–5 | Clarify: tokenization, tie-breaking, what "relevant" means |
| 5–17 | Redo **Drill 4.2** (BM25) |
| 17–27 | Redo **Drill 4.3** (RRF) and explain aloud why you fuse ranks, not scores |
| 27–42 | **New problem C** below: recall@k, MRR, and nDCG@k |
| 42–45 | Answer **Q41** (hybrid search) and **Q43** (chunk size) out loud |

#### New problem C — Retrieval metrics: recall@k, MRR, nDCG@k 🟡 (15 min)

**Problem.** With binary relevance, implement `recall_at_k(ranked, relevant, k)` (fraction of **all** relevant documents found in the top k), `mean_reciprocal_rank(rankings, relevant_sets)` (average of 1/position of the first relevant hit, 0 if none), and `ndcg_at_k(ranked, relevant, k)`:

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i + 1)}, \qquad \text{nDCG@k} = \frac{\text{DCG@k}}{\text{DCG@k of the ideal ranking}}$$

In [55]:
from sklearn.metrics import ndcg_score

ranked_q1, relevant_q1 = ["d3", "d1", "d7", "d2", "d9"], {"d1", "d2", "d8"}      # d8 is relevant but was never retrieved
ranked_q2, relevant_q2 = ["d5", "d6", "d4"], {"d4"}
all_q1 = ranked_q1 + ["d8"]                                                     # give the missed doc the lowest score
sk_ndcg5 = ndcg_score([[1.0 if doc in relevant_q1 else 0.0 for doc in all_q1]], [[6, 5, 4, 3, 2, 1]], k=5)
print(f"sklearn nDCG@5 for query 1: {sk_ndcg5:.4f}")

sklearn nDCG@5 for query 1: 0.4982


In [56]:
def recall_at_k(ranked, relevant, k):
    return None  # TODO


def mean_reciprocal_rank(rankings, relevant_sets):
    return None  # TODO


def ndcg_at_k(ranked, relevant, k):
    return None  # TODO: binary relevance


check("recall@3", recall_at_k(ranked_q1, relevant_q1, 3), 1 / 3)
check("recall@5", recall_at_k(ranked_q1, relevant_q1, 5), 2 / 3, hint="Divide by ALL relevant documents, including never-retrieved ones.")
check("MRR over two queries", mean_reciprocal_rank([ranked_q1, ranked_q2], [relevant_q1, relevant_q2]), (1 / 2 + 1 / 3) / 2)
check("nDCG@5 vs sklearn", ndcg_at_k(ranked_q1, relevant_q1, 5), sk_ndcg5,
      hint="The ideal ranking puts min(len(relevant), k) relevant documents first.")
check("nDCG@5 by hand", ndcg_at_k(ranked_q1, relevant_q1, 5), (1 / np.log2(3) + 1 / np.log2(5)) / (1 + 1 / np.log2(3) + 1 / np.log2(4)))

⏳ recall@3: not attempted yet — replace None with your answer.
⏳ recall@5: not attempted yet — replace None with your answer.
⏳ MRR over two queries: not attempted yet — replace None with your answer.
⏳ nDCG@5 vs sklearn: not attempted yet — replace None with your answer.
⏳ nDCG@5 by hand: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / len(relevant) if relevant else 0.0


def mean_reciprocal_rank(rankings, relevant_sets):
    reciprocal_ranks = [next((1.0 / pos for pos, doc in enumerate(ranked, start=1) if doc in relevant), 0.0)
                        for ranked, relevant in zip(rankings, relevant_sets)]
    return float(np.mean(reciprocal_ranks))


def ndcg_at_k(ranked, relevant, k):
    gains = np.array([1.0 if doc in relevant else 0.0 for doc in ranked[:k]])
    dcg = np.sum(gains / np.log2(np.arange(2, len(gains) + 2)))
    ideal_hits = min(len(relevant), k)
    idcg = np.sum(1.0 / np.log2(np.arange(2, ideal_hits + 2)))
    return float(dcg / idcg) if idcg > 0 else 0.0


check("recall@3", recall_at_k(ranked_q1, relevant_q1, 3), 1 / 3)
check("recall@5", recall_at_k(ranked_q1, relevant_q1, 5), 2 / 3)
check("MRR over two queries", mean_reciprocal_rank([ranked_q1, ranked_q2], [relevant_q1, relevant_q2]), (1 / 2 + 1 / 3) / 2)
check("nDCG@5 vs sklearn", ndcg_at_k(ranked_q1, relevant_q1, 5), sk_ndcg5)
check("nDCG@5 by hand", ndcg_at_k(ranked_q1, relevant_q1, 5), (1 / np.log2(3) + 1 / np.log2(5)) / (1 + 1 / np.log2(3) + 1 / np.log2(4)))
```

**Complexity:** O(k + |relevant|) per query with set lookups.
</details>

**Q50. Recall@k, MRR, or nDCG — which metric fits which retrieval use case?**

<details><summary>Show answer</summary>

- **30-second answer:** Recall@k asks "did the relevant documents make it into the k results the LLM will see?" — the key metric for a RAG retrieval stage. MRR only looks at the first relevant hit, so it suits "one good answer is enough" (FAQ lookup, navigational search). nDCG@k rewards putting *all* relevant — and more relevant — documents near the top with a logarithmic position discount; it's the standard for ranked search (BEIR and MTEB report nDCG@10).
- **Go deeper:** Evaluate retrieval and generation separately: with poor recall@k, no prompt can rescue the answer. Build a labelled query set from real traffic, report metrics per query segment, and watch for "relevant but unlabelled" documents that make metrics look worse than reality.
- **❌ Common wrong answer:** "Just judge the final LLM answers — retrieval metrics are unnecessary."

</details>

## 📋 Answer Framework & Cheat Sheet

### The 7-step framework for any coding question

| Step | What you do | What it sounds like |
|---|---|---|
| **1. Clarify** | Inputs, shapes, dtypes, allowed libraries, batching, edge cases | "Logits are `(n, C)` floats and labels ints in 0..C−1? NumPy only?" |
| **2. Examples** | One tiny hand-checkable example + one edge case | "For [1, 2, 3] I expect about [0.09, 0.24, 0.67]; then I'll try logits of 1000." |
| **3. Approach** | Name the algorithm, the vectorization trick, the stability trick, the complexity | "Shift by the max, log-sum-exp for the loss, O(n·C)." |
| **4. Code** | Signature → shape comments → core → edge cases | "`keepdims=True` so the division broadcasts per row." |
| **5. Test** | Run the example, the edge case, compare with a library or brute force | "Matches `scipy.special.softmax`; no `nan` for large logits." |
| **6. Complexity** | Time and memory in n, d, k, T; name the bottleneck | "The `(q, n)` similarity matrix dominates memory — I'd batch queries." |
| **7. Extensions** | Scale, numerics, production concerns | "For 100M documents, an HNSW index plus a reranker." |

### Formulas to write from memory

| Topic | Formula / rule |
|---|---|
| Stable softmax | $e^{z - \max z} / \sum_j e^{z_j - \max z}$ |
| Log-sum-exp | $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$, with $m = \max_j z_j$ |
| Softmax cross-entropy gradient | $\partial L/\partial z = (\text{softmax}(z) - \text{onehot}(y))/n$ |
| Binary cross-entropy from logits | $\log(1 + e^z) - yz$ → `np.logaddexp(0, z) - y*z` |
| MSE gradient | $\frac{2}{n} X^\top (Xw - y)$; GD stable for lr $< 2/\lambda_{\max}(\frac{2}{n}X^\top X)$ |
| Logistic regression gradient | $\frac{1}{n} X^\top(\sigma(Xw + b) - y) + \lambda w$; scikit-learn's $C$ ↔ $\lambda = 1/(C\,n)$ |
| Squared distances | $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a^\top b$ (clip at 0) |
| Cosine similarity | $a^\top b / (\lVert a\rVert\,\lVert b\rVert)$; unit vectors: $\lVert a-b\rVert^2 = 2 - 2\cos$ |
| k-means / k-means++ | minimize $\sum_i \lVert x_i - \mu_{c(i)}\rVert^2$; next seed with probability ∝ $D(x)^2$ |
| Gini / entropy | $1 - \sum_c p_c^2$ / $-\sum_c p_c \log_2 p_c$ |
| Gaussian log-density | $-\frac{1}{2}\left[\log(2\pi\sigma^2) + (x - \mu)^2/\sigma^2\right]$ |
| PCA | $X_c = U S V^\top$; components = rows of $V^\top$; variance $S^2/(n-1)$ |
| Precision / recall / F1 | $\frac{TP}{TP+FP}$ · $\frac{TP}{TP+FN}$ · $\frac{2TP}{2TP+FP+FN}$ |
| ROC-AUC from ranks | $\left(\sum_{\text{pos}} r_i - n_p(n_p+1)/2\right) / (n_p\,n_n)$ |
| Conv output size / params | $\lfloor (H + 2p - k)/s \rfloor + 1$ / $F \cdot C \cdot k_h k_w + F$ |
| Batch norm | $\gamma\,(x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon} + \beta$; eval uses running statistics |
| Inverted dropout | $x \cdot m / (1 - p)$ with $m \sim \text{Bernoulli}(1 - p)$ |
| Adam / AdamW | $\theta \leftarrow \theta - \text{lr}\,\hat m / (\sqrt{\hat v} + \varepsilon)$, $\hat m = m/(1-\beta_1^t)$; AdamW first does $\theta \leftarrow \theta(1 - \text{lr}\,\lambda)$ |
| Attention | $\text{softmax}(QK^\top/\sqrt{d_k} + M)\,V$, $M = -\infty$ where blocked |
| Top-p | keep a token if the probability of higher-ranked tokens is $< p$ |
| BM25 | $\sum_t \text{idf}(t)\,\frac{tf\,(k_1 + 1)}{tf + k_1(1 - b + b\,\lvert d\rvert/\text{avgdl})}$ |
| RRF | $\sum_{\text{lists}} 1/(k + \text{rank})$, $k = 60$ |
| nDCG@k | $\sum_{i \le k} \text{rel}_i / \log_2(i + 1)$, divided by the ideal DCG |
| IoU | $\text{area}(A \cap B) / (\text{area}(A) + \text{area}(B) - \text{area}(A \cap B))$ |
| Token bucket | tokens ← min(capacity, tokens + Δt · rate); allow if tokens ≥ cost |

### Numerically stable tricks

| Situation | ❌ Fragile | ✅ Stable |
|---|---|---|
| Softmax | `np.exp(z) / np.exp(z).sum()` | subtract `z.max(axis, keepdims=True)` first |
| Log-probabilities | `np.log(softmax(z))` | log-softmax / `scipy.special.logsumexp` |
| Sigmoid for very negative z | `1 / (1 + np.exp(-z))` | branch on the sign, or `scipy.special.expit` |
| Binary cross-entropy | `-y*log(p) - (1-y)*log(1-p)` | from logits: `np.logaddexp(0, z) - y*z` |
| Products of probabilities | `np.prod(p)` | `np.sum(np.log(p))` |
| Distances | `np.sqrt(sq)` | `np.sqrt(np.maximum(sq, 0))` |
| Normalizing | `x / norm`, `x / std` | `x / np.maximum(norm, eps)`; std = 0 → 1 |
| Linear systems | `np.linalg.inv(A) @ b` | `np.linalg.solve(A, b)` / `np.linalg.lstsq` |
| Attention masks | a hard-coded `-1e9` (doesn't fit in float16) | `-np.inf` or `torch.finfo(dtype).min` |
| Gradient checks | float32 with ε = 1e-10 | float64, central differences, ε ≈ 1e-6 |

### Testing checklist (say these out loud)

- [ ] A tiny example you can verify by hand
- [ ] Shapes asserted at the boundaries of your function
- [ ] Edge cases: empty input, k = 1 and k = n, ties, a class with no samples, an all-zero vector, huge and tiny logits
- [ ] Comparison against a trusted library or a brute-force loop on random data
- [ ] A fixed random seed so failures are reproducible

## 📚 Resources

### 📖 Official Docs
- [numpy.lib.stride_tricks.sliding_window_view — NumPy Manual](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html) — zero-copy windows and patches (moving averages, im2col)
- [roc_auc_score — scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html) — the reference the AUC drill is checked against
- [ndcg_score — scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ndcg_score.html) — graded relevance and the `k` cutoff

### 🎥 Videos
- [Andrej Karpathy — The spelled-out intro to neural networks and backpropagation: building micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) (2 h 25 min) — backprop derived by hand, then a tiny autograd engine; the exact reasoning Drill 3.1 tests
- [Andrej Karpathy — Let's build GPT: from scratch, in code, spelled out.](https://www.youtube.com/watch?v=kCc8FmEb1nY) (1 h 56 min) — attention, causal masking, and sampling written live; pairs with Drills 3.6–3.7
- [Andrej Karpathy — Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) (2 h 13 min) — BPE training and encoding, byte-level details, and tokenizer quirks; pairs with Drill 4.1

### 📄 Papers
- [Kingma & Ba (2014) — Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980)
- [Loshchilov & Hutter (2017) — Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) — AdamW
- [Ioffe & Szegedy (2015) — Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167)
- [Srivastava et al. (2014) — Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html)
- [Vaswani et al. (2017) — Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Dao et al. (2022) — FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135)
- [Sennrich et al. (2016) — Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909) — BPE for NLP
- [Holtzman et al. (2019) — The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) — nucleus (top-p) sampling
- [Arthur & Vassilvitskii (2007) — k-means++: The Advantages of Careful Seeding (PDF)](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf)
- [Cormack, Clarke & Büttcher (2009) — Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods (PDF)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)

### 📘 Books & Courses
- [Chip Huyen — Introduction to Machine Learning Interviews Book](https://huyenchip.com/ml-interviews-book/) — free; interview process, question bank, and how rounds are evaluated
- [Andrej Karpathy — Neural Networks: Zero To Hero](https://karpathy.ai/zero-to-hero.html) — the full lecture series behind the videos above
- [CS231n Deep Learning for Computer Vision — Convolutional Networks notes](https://cs231n.github.io/convolutional-networks/) — output sizes, parameter counts, and im2col
- [CS231n Deep Learning for Computer Vision — Neural Networks Part 3 notes](https://cs231n.github.io/neural-networks-3/) — gradient checks, sanity checks, and optimizers

### 🏋️ Practice
- [Deep-ML | Practice Machine Learning](https://www.deep-ml.com/) — LeetCode-style ML and deep-learning implementation problems with test cases
- [GitHub - Exorust/TorchLeet](https://github.com/Exorust/TorchLeet) — PyTorch implementation exercises from basic to LLM-level
- [GitHub - rougier/numpy-100: 100 numpy exercises (with solutions)](https://github.com/rougier/numpy-100) — vectorization fluency
- LeetCode problem 146 "LRU Cache" — the classic version of Drill 4.5 with a large hidden test suite

## ➡️ What's Next

**[02 · ML Theory & Statistics Questions](02_ML_Theory_and_Statistics_Questions.ipynb)** — coding rounds test whether you can *build* the algorithms; theory screens test whether you can *explain* why they work, when they fail, and how to evaluate them — probability, statistics, bias–variance, deep learning, LLMs, and MLOps, with runnable demonstrations that make the answers stick.